In [0]:
# Write to catalog
write = False
aggregationLevel = ["L1"]
withMissingColumns = True

daily_aggregated_table = "mdl_ui_xpl_dev.ai_sandbox.Daily_aggregated_table"
monthly_aggregated_table = "mdl_ui_xpl_dev.ai_sandbox.MONTHLY_aggregated_table"
MONTHLY_DEVIATIONS_TABLE = "mdl_ui_xpl_dev.ai_sandbox.MONTHLY_DEVIATIONS_TABLE"
monthly_deviation_table_buckets = "mdl_ui_xpl_dev.ai_sandbox.monthly_deviation_table_buckets"
anomaly_list_table = "mdl_ui_xpl_dev.ai_sandbox.anomaly_list_table"
anomaly_case_details_table = "mdl_ui_xpl_dev.ai_sandbox.anomaly_case_details_table"
union_anomaly_case_details_table = "mdl_ui_xpl_dev.ai_sandbox.union_anomaly_case_details_table"

target_month = "2026-02-01"    # Filter month maybe should be calculated in App? (All handles via app, add a table for it)
severity_count_threshold = 3   # The threshhold that definds how many rules are over it per customer, used for creating the column f'{rules_above_threshold} / {total_number_of_rules}'



##Load initial table to this session

In [0]:
%sql
-- Save the table in baseline_table for use in other cell
create or replace temporary view initiale_table as
select distinct
  SALES_DOCUMENT,
  DOCUMENT_DATE,
  order_table.MATERIAL_CODE as MATERIAL_CODE,
  ORDER_QUANTITY,
  order_table.MATERIAL_DESCRIPTION,
  order_table.SOLD_TO_CODE,
  split(hierarchy_table.KPI_HIERARCHY_LEVEL_1_CODE, '-')[0] as L1,
  split(hierarchy_table.KPI_HIERARCHY_LEVEL_2_CODE, '-')[1] as L2,
  split(hierarchy_table.KPI_HIERARCHY_LEVEL_3_CODE, '-')[2] as L3,
  CUSTOMER_HIERARCHY_CODE,
  SOLD_TO_CUSTOMER_NAME,
  NET_PRICE,
  NET_PRICE_EUR,
  DOCUMENT_CURRENCY,
  DETAILS_NET_VALUE,
  LIST_PRICE,
  NET_PRICE_EUR * ORDER_QUANTITY as NET_ITEM_VALUE_EUR,
  -- CS_NET_WEIGHT,
  -- CS_NET_WEIGHT_UNIT,
  -- PC_PER_CS,
  -- PC_PER_PAL,
  BRAND_DESCRIPTION,
  BRAND_CODE,
  CATEGORY_DESCRIPTION,
  CATEGORY_CODE,
  SECTOR_DESCRIPTION as FORMAT_DESCRIPTION,
  SECTOR_CODE as FORMAT_CODE,
  order_table.SALES_DOCUMENT_TYPE,
  order_table.SALES_ORGANIZATION,
  MATERIAL_TYPE
  -- order_table.DELETION_INDICATOR

from mdl_ui_xpl_dev.gold.g_u2k2_uapl_sales_orders_not_automated as order_table
-- left join mdl_ui_xpl_dev.gold.g_u2k2_uapl_material_uom_not_automated as product_table
--   on order_table.MATERIAL_CODE = product_table.MATERIAL_CODE
left join mdl_ui_xpl_dev.gold.g_u2k2_uapl_material_master_not_automated as product_master_table
  on order_table.MATERIAL_CODE = product_master_table.MATERIAL_CODE
left join mdl_ui_xpl_dev.gold.anaplan_uapl_customer_hierarchy_not_automated as hierarchy_table
  on order_table.SOLD_TO_CODE = hierarchy_table.SOLD_TO_CODE;

select * from initiale_table
where ORDER_QUANTITY > 0 and SALES_DOCUMENT_TYPE not in ("B1", "B3", "B4") and SALES_ORGANIZATION in ("0I01" ,"0I02") and MATERIAL_TYPE = "FERT" and SALES_DOCUMENT_TYPE = "ZTA"

%md
# Daily aggregated table

In [0]:
############## Import librarys #################
import math
from pyspark.sql import functions as F
from pyspark.sql.functions import col, row_number, concat_ws, round, when, lag, datediff, struct, collect_set, map_from_entries, lit, sum, year, month, sequence, explode, min as min_, max as max_, lit, months_between, floor, col
from pyspark.sql.window import Window
from pyspark.sql.types import DoubleType, StringType

In [0]:
############### Setup, Filter, and Formatting ################

# Load data and determine AggLevel
df = spark.table("initiale_table")
dfAgg = spark.table("mdl_ui_xpl_dev.ai_sandbox.hierarchy_indicator_table")

AggLevel = dfAgg.select("AGGREGATE_BY").collect()[0][0]
AggLevel = AggLevel if AggLevel in ["L1", "L2", "L3"] else "SOLD_TO_CODE"   # Notice this is not a list of levels currently
AggLevel = aggregationLevel # Currently overwrites the imported table 
# AggLevel = ["L1","L2"]
print(f"Aggregation Level: {AggLevel}")

# Filter and format date
df = df.withColumn("EVENT_DATE", F.to_date(col("DOCUMENT_DATE")))
df = df.filter(
    (col("DOCUMENT_DATE") >= "2022-01-01"))

In [0]:
##################### Daily aggregation & Brand format mix(quantity) ######################
results = []
for x in AggLevel:
    w_daily_brand_format = Window.partitionBy(x, "EVENT_DATE", "BRAND_CODE", "FORMAT_CODE")
    df_tmp = df.withColumn("BRAND_FORMAT_SUM", F.round(sum("ORDER_QUANTITY").over(w_daily_brand_format), 0).cast("int"))
    df_tmp = df_tmp.withColumn("BRAND_FORMAT_KEY", concat_ws("_", col("BRAND_DESCRIPTION"), col("FORMAT_DESCRIPTION")))

    # Daily aggregation
    agg_df = df_tmp.groupBy(col(x).alias("CUSTOMER_CODE"), "EVENT_DATE").agg(
        sum("NET_ITEM_VALUE_EUR").alias("TOTAL_VALUE_OF_DAILY_ORDER_EUR"),
        sum("ORDER_QUANTITY").alias("TOTAL_QUANTITY_OF_DAILY_ORDER"),
        map_from_entries(collect_set(struct("BRAND_FORMAT_KEY", "BRAND_FORMAT_SUM"))).alias("BRAND_FORMAT_MIX"),
        F.first("CUSTOMER_HIERARCHY_CODE").alias("CUSTOMER_HIERARCHY_CODE"),
    )
    agg_df = agg_df.withColumn("AGGREGATE_BY", F.lit(x))
    results.append(agg_df)
# Union all levels
daily_agg_combined = results[0]
for d in results[1:]:
    daily_agg_combined = daily_agg_combined.unionByName(d)
# agg_df is now all levels combined
agg_df = daily_agg_combined
# display(agg_df.orderBy("CUSTOMER_CODE","EVENT_DATE"))

In [0]:
########################## EVENT_IDENTIFIER ##########################
w_event = Window.partitionBy("AGGREGATE_BY").orderBy("EVENT_DATE")
agg_df = agg_df.withColumn("EVENT_IDENTIFIER", row_number().over(w_event))

############################ AGGREGATE_BY ###########################
# agg_df = agg_df.withColumn("AGGREGATE_BY", F.col("AGGREGATE_BY"))

###################### TOTAL_TIME_FROM_LAST_ORDER ####################
w_user = Window.partitionBy("CUSTOMER_CODE", "AGGREGATE_BY").orderBy("EVENT_DATE")
agg_df = agg_df.withColumn("TOTAL_TIME_FROM_LAST_ORDER", datediff(col("EVENT_DATE"), lag("EVENT_DATE").over(w_user)))


In [0]:
################ Final Selection and Display ################
agg_df_final = agg_df.select(
    "EVENT_IDENTIFIER",
    "AGGREGATE_BY",
    #"SOLD_TO_CODE​",
    "CUSTOMER_HIERARCHY_CODE",
    "CUSTOMER_CODE",
    "EVENT_DATE",
    "TOTAL_TIME_FROM_LAST_ORDER",
    #"NET_ITEM_VALUE_EUR​",
    "TOTAL_VALUE_OF_DAILY_ORDER_EUR",
    "TOTAL_QUANTITY_OF_DAILY_ORDER",
    "BRAND_FORMAT_MIX",
)
if write:   
    agg_df_final.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(daily_aggregated_table)
display(agg_df_final.orderBy("AGGREGATE_BY","CUSTOMER_CODE","EVENT_DATE"))

## monthly_aggregated_table

In [0]:
################## Baseline Date (Multi-Level) ##################

# Extract year and month from EVENT_DATE (keep AGGREGATE_BY)
agg_df_month = agg_df_final.withColumn(
    "BASELINE_DATE",
    F.to_date(F.concat_ws("-", year(col("EVENT_DATE")), month(col("EVENT_DATE")), F.lit("01")), "yyyy-M-d")
)

In [0]:
################# Main Monthly Aggregations (Multi-Level) #####################
# Perform aggregation by CUSTOMER_CODE, AGGREGATE_BY, YEAR, MONTH
agg_df_final_month = agg_df_month.groupBy(
    "AGGREGATE_BY", "CUSTOMER_CODE", "BASELINE_DATE"
).agg(
    F.count("*").alias("N_EVENTS_MONTH"),
    F.avg("TOTAL_TIME_FROM_LAST_ORDER").alias("AVG_TOTAL_TIME_FROM_LAST_ORDER"),
    F.expr("percentile_approx(TOTAL_TIME_FROM_LAST_ORDER, 0.5)").alias("MEDIAN_TOTAL_TIME_FROM_LAST_ORDER"),
    F.sum("TOTAL_VALUE_OF_DAILY_ORDER_EUR").alias("TOTAL_ORDER_VALUE_EUR"),
    F.sum("TOTAL_QUANTITY_OF_DAILY_ORDER").alias("TOTAL_ORDER_QUANTITY"),
).filter(col("BASELINE_DATE") <= target_month)  # Remove non relevent dates

In [0]:
################## Calculate Monthly Mix and Fractions (Multi-Level) ######################
# Window to get the total monthly quantity per customer (and AGGREGATE_BY)
w_month = Window.partitionBy("AGGREGATE_BY", "CUSTOMER_CODE", "BASELINE_DATE")

# Calculate both quantity and fraction maps
monthly_mix_map = agg_df_month.select(
    "AGGREGATE_BY", "CUSTOMER_CODE", "BASELINE_DATE", 
    F.explode("BRAND_FORMAT_MIX").alias("BRAND_KEY", "DAILY_QTY")
).groupBy("AGGREGATE_BY", "CUSTOMER_CODE", "BASELINE_DATE", "BRAND_KEY").agg(
    F.sum("DAILY_QTY").alias("MONTHLY_QTY"),
).withColumn(
    "MONTHLY_FRACTION", 
    F.when(F.sum("MONTHLY_QTY").over(w_month) == 0, 0)
     .otherwise(F.round(col("MONTHLY_QTY") / F.sum("MONTHLY_QTY").over(w_month), 4))
).groupBy("AGGREGATE_BY", "CUSTOMER_CODE", "BASELINE_DATE").agg(
    F.map_from_entries(F.collect_list(F.struct("BRAND_KEY", "MONTHLY_QTY"))).alias("TOTAL_BRAND_FORMAT_MIX"),
    F.map_from_entries(F.collect_list(F.struct("BRAND_KEY", "MONTHLY_FRACTION"))).alias("FRACTION_BRAND_FORMAT_MIX")
)
# Join it back
agg_df_final_month = agg_df_final_month.join(monthly_mix_map, on=["AGGREGATE_BY", "CUSTOMER_CODE", "BASELINE_DATE"], how="left")

In [0]:
########### Add EVENT_IDENTIFIER as a unique key (Multi-Level) #################
w_event = Window.partitionBy("AGGREGATE_BY").orderBy("CUSTOMER_CODE", "BASELINE_DATE")
agg_df_final_month = agg_df_final_month.withColumn("EVENT_IDENTIFIER", row_number().over(w_event))

In [0]:
#####################  Selection & Display (Multi-Level) ###################### Thit table does not include months which do not exist
# Final select
agg_df_final_month = agg_df_final_month.select(
    "EVENT_IDENTIFIER",
    "AGGREGATE_BY",
    "CUSTOMER_CODE",
    "BASELINE_DATE",
    "N_EVENTS_MONTH",
    "AVG_TOTAL_TIME_FROM_LAST_ORDER",
    "MEDIAN_TOTAL_TIME_FROM_LAST_ORDER",
    "TOTAL_ORDER_VALUE_EUR",
    "TOTAL_ORDER_QUANTITY",
    "TOTAL_BRAND_FORMAT_MIX",
    "FRACTION_BRAND_FORMAT_MIX"
)
# if write: 
#     agg_df_final_month.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(monthly_aggregated_table)
# display(agg_df_final_month) 

In [0]:
# ------------------------ Add for each customer rows of which he does not have existing months --------------------------
if(withMissingColumns):
    # Get min and max BASELINE_DATE per customer and AGGREGATE_BY
    date_range_df = agg_df_final_month.groupBy("CUSTOMER_CODE", "AGGREGATE_BY").agg(
        min_("BASELINE_DATE").alias("min_date"),
        F.lit(target_month).alias("max_date")
    )

    # Generate all months between min and max for each customer and AGGREGATE_BY
    date_range_df = date_range_df.withColumn(
        "all_months",
        sequence(
            "min_date",
            "max_date",
            lit(1).cast("interval month")
        )
    ).select(
        "CUSTOMER_CODE",
        "AGGREGATE_BY",
        explode("all_months").alias("BASELINE_DATE")
    )

    # Join with original data
    agg_df_final_month_filled = date_range_df.join(
        agg_df_final_month,
        on=["CUSTOMER_CODE", "AGGREGATE_BY", "BASELINE_DATE"],
        how="left"
    )

    # Fill numeric columns with 0
    numeric_cols = [
        "N_EVENTS_MONTH",
        "AVG_TOTAL_TIME_FROM_LAST_ORDER",
        "MEDIAN_TOTAL_TIME_FROM_LAST_ORDER",
        "TOTAL_ORDER_VALUE_EUR",
        "TOTAL_ORDER_QUANTITY"
    ]

    for col_name in numeric_cols:
        if col_name in ["AVG_TOTAL_TIME_FROM_LAST_ORDER", "MEDIAN_TOTAL_TIME_FROM_LAST_ORDER"]:
            agg_df_final_month_filled = agg_df_final_month_filled.withColumn(
                col_name,
                F.coalesce(F.col(col_name), F.lit(30))  # For frequancy only
            )
        else:
            agg_df_final_month_filled = agg_df_final_month_filled.withColumn(
                col_name,
                F.coalesce(F.col(col_name), F.lit(0))
            )

    # Fill map columns with empty map
    map_cols = ["TOTAL_BRAND_FORMAT_MIX", "FRACTION_BRAND_FORMAT_MIX"]

    for col_name in map_cols:
        agg_df_final_month_filled = agg_df_final_month_filled.withColumn(
            col_name,
            F.when(col(col_name).isNull(), F.create_map()).otherwise(col(col_name))
        )

    # Fill EVENT_IDENTIFIER with 0
    agg_df_final_month_filled = agg_df_final_month_filled.withColumn(
        "EVENT_IDENTIFIER",
        F.coalesce(col("EVENT_IDENTIFIER"), F.lit(0))
    )

    agg_df_final_month_filled = agg_df_final_month_filled.withColumn(
        "IS_FILLED_ROW",
        F.when(col("N_EVENTS_MONTH") == 0, F.lit(1)).otherwise(F.lit(0))
    )
    display(agg_df_final_month_filled.orderBy("CUSTOMER_CODE", "AGGREGATE_BY", "BASELINE_DATE"))

In [0]:
#------------------ History data import --------------------

# Import the History data
dfAgg = spark.table("mdl_ui_xpl_dev.ai_sandbox.reference_history_definition_table")
row = dfAgg.select(
    "REQUIRED_HISTORY_LENGTH",
    "MINIMAL_REQUIRED_HISTORY_LENGTH"
).collect()[0]

REQUIRED_HISTORY_LENGTH = row["REQUIRED_HISTORY_LENGTH"]    # 12
MINIMAL_REQUIRED_HISTORY_LENGTH = row["MINIMAL_REQUIRED_HISTORY_LENGTH"] # 8
print(REQUIRED_HISTORY_LENGTH, MINIMAL_REQUIRED_HISTORY_LENGTH)


# Create new table & Add the history data
if(withMissingColumns):
    df_monthly_deviations = agg_df_final_month_filled.withColumn("REQUIRED_HISTORY_LENGTH", lit(REQUIRED_HISTORY_LENGTH)) \
                                          .withColumn("MINIMAL_REQUIRED_HISTORY_LENGTH", lit(MINIMAL_REQUIRED_HISTORY_LENGTH)) \
                                          .withColumn("MONTH_IDX", F.year("BASELINE_DATE") * 12 + F.month("BASELINE_DATE"))
else:
    df_monthly_deviations = agg_df_final_month.withColumn("REQUIRED_HISTORY_LENGTH", lit(REQUIRED_HISTORY_LENGTH)) \
                                          .withColumn("MINIMAL_REQUIRED_HISTORY_LENGTH", lit(MINIMAL_REQUIRED_HISTORY_LENGTH)) \
                                          .withColumn("MONTH_IDX", F.year("BASELINE_DATE") * 12 + F.month("BASELINE_DATE"))

# display(df_monthly_deviations)

In [0]:
# #---------------------- Rolling windows ----------------------
w_brand_format = Window.partitionBy("CUSTOMER_CODE", "BRAND_FORMAT_KEY").orderBy("MONTH_IDX").rangeBetween(-REQUIRED_HISTORY_LENGTH, -1)
w_general = Window.partitionBy("CUSTOMER_CODE").orderBy("MONTH_IDX").rangeBetween(-REQUIRED_HISTORY_LENGTH, -1)

# ----------------- Is there the minimum history that needed --------------
df_monthly_deviations = df_monthly_deviations.withColumn(
    "HIST_EVENT_COUNT",
    F.count(F.lit(1)).over(w_general)  # How many months were involved in this time period
).withColumn(
    "MONTHS_SINCE_FIRST_EVENT",
    F.col("MONTH_IDX") - F.min("MONTH_IDX").over(Window.partitionBy("CUSTOMER_CODE"))
).withColumn(   # Has at least 8 months(Allowed to calculate)
    "HAS_REQUIRED_HISTORY",
    F.col("MONTHS_SINCE_FIRST_EVENT") >= MINIMAL_REQUIRED_HISTORY_LENGTH
).withColumn(   # Has 12 months since(stronger argument - for knowing if to add 0's to the calculations)
    "HAS_MORE_THEN_REQUIRED_HISTORY_LENGTH",
    F.col("MONTHS_SINCE_FIRST_EVENT") >= REQUIRED_HISTORY_LENGTH
).withColumn(
    "REMAINING_MONTHS_FOR_CALCULATIONS",
    F.when((F.col("HAS_MORE_THEN_REQUIRED_HISTORY_LENGTH")), F.lit(REQUIRED_HISTORY_LENGTH) - F.col("HIST_EVENT_COUNT"))    # If there are 12 months available then find remaining and fill 0's
    .otherwise(0)   # otherwise there are less then 12 and so i want to calculate only what exists (if less then MINIMAL_REQUIRED_HISTORY_LENGTH then it wont be calculated to begin with)
)



# display(df_monthly_deviations)
    
    # minimum of MINIMAL_REQUIRED_HISTORY_LENGTH from the first order month of a customer


In [0]:
# ----------------- HISTORICAL_FRACTION_BRAND_FORMAT_MIX -------------------

# Long table: one row per customer-month-brand_format
base_df = (
    df_monthly_deviations
    .select(
        "CUSTOMER_CODE",
        "BASELINE_DATE",
        "HAS_REQUIRED_HISTORY",
        F.explode("TOTAL_BRAND_FORMAT_MIX").alias("BRAND_FORMAT_KEY", "MONTHLY_QTY")
    )
    .groupBy("CUSTOMER_CODE", "BASELINE_DATE", "HAS_REQUIRED_HISTORY", "BRAND_FORMAT_KEY")
    .agg(F.sum("MONTHLY_QTY").alias("MTH_QTY"))
    .withColumn("MONTH_IDX", F.year("BASELINE_DATE") * 12 + F.month("BASELINE_DATE"))
)

# All customer-months
customer_months = (
    base_df
    .select("CUSTOMER_CODE", "BASELINE_DATE", "MONTH_IDX", "HAS_REQUIRED_HISTORY")
    .distinct()
)

# All brand formats ever seen per customer
customer_keys = (
    base_df
    .select("CUSTOMER_CODE", "BRAND_FORMAT_KEY")
    .distinct()
)

# Dense grid: every customer-month-key combo
dense_df = (
    customer_months
    .join(customer_keys, on="CUSTOMER_CODE", how="inner")
    .join(
        base_df.select("CUSTOMER_CODE", "BASELINE_DATE", "MONTH_IDX", "BRAND_FORMAT_KEY", "MTH_QTY"),
        on=["CUSTOMER_CODE", "BASELINE_DATE", "MONTH_IDX", "BRAND_FORMAT_KEY"],
        how="left"
    )
    .fillna({"MTH_QTY": 0})
)

scored_df = (
    dense_df
    .withColumn(
        "ROLL_QTY", 
        F.when(
            F.col("HAS_REQUIRED_HISTORY"),
            F.sum("MTH_QTY").over(w_brand_format))
        )
    .withColumn(
        "ROLL_TOTAL", 
        F.when(
            F.col("HAS_REQUIRED_HISTORY"),
            F.sum("MTH_QTY").over(w_general))
        )
    .filter(F.col("ROLL_QTY") > 0)
    .withColumn("ROLL_FRAC", F.round(F.col("ROLL_QTY") / F.col("ROLL_TOTAL"), 4))
)
    
# monthly_mix_map = (
#     scored_df
#     .groupBy("CUSTOMER_CODE", "BASELINE_DATE")
#     .agg(
#         F.map_from_entries(
#             F.collect_list(F.struct("BRAND_FORMAT_KEY", "ROLL_FRAC"))
#         ).alias("HISTORICAL_FRACTION_BRAND_FORMAT_MIX")
#     )
# )
monthly_mix_map = (
    scored_df
    .groupBy("CUSTOMER_CODE", "BASELINE_DATE")
    .agg(
        F.map_from_entries(
            F.collect_list(F.struct("BRAND_FORMAT_KEY", "ROLL_FRAC"))
        ).alias("HISTORICAL_FRACTION_BRAND_FORMAT_MIX"),

        F.map_from_entries(
            F.collect_list(F.struct("BRAND_FORMAT_KEY", "ROLL_QTY"))
        ).alias("HISTORICAL_TOTAL_BRAND_FORMAT_MIX")
    )
)

df_monthly_deviations = df_monthly_deviations.drop("HISTORICAL_FRACTION_BRAND_FORMAT_MIX")
df_monthly_deviations = df_monthly_deviations.join(
    monthly_mix_map,
    on=["CUSTOMER_CODE", "BASELINE_DATE"],
    how="left"
)

# display(df_monthly_deviations.orderBy("CUSTOMER_CODE","BASELINE_DATE"))

In [0]:
# ------------------------- BRAND_FORMAT_MIX_SIMILARITY --------------------------
@F.udf(DoubleType())
def get_cosine_sim(mix1, mix2):
    # Strictly ensure both are dictionaries before proceeding
    if not isinstance(mix1, dict) or not isinstance(mix2, dict): 
        return None
    
    all_keys = set(mix1.keys()).union(set(mix2.keys()))
    dot_product = math.fsum(mix1.get(k, 0.0) * mix2.get(k, 0.0) for k in all_keys)
    
    norm1 = math.sqrt(math.fsum(v**2 for v in mix1.values()))
    norm2 = math.sqrt(math.fsum(v**2 for v in mix2.values()))
    
    return dot_product / (norm1 * norm2) if (norm1 * norm2) > 0 else 0.0

# Apply it to your dataframe
df_monthly_deviations = df_monthly_deviations.withColumn(
    "BRAND_FORMAT_MIX_SIMILARITY", 
    get_cosine_sim(F.col("FRACTION_BRAND_FORMAT_MIX"), F.col("HISTORICAL_FRACTION_BRAND_FORMAT_MIX"))
)

In [0]:
#------------------------------ SIGNS ----------------------------------
df_monthly_deviations_signes = df_monthly_deviations \
.withColumn( # ---------------- FREQUENCY ----------------
    "SIGN_AVG_FREQUENCY_CHANGE",
    F.when(
        (F.col("AVG_TOTAL_TIME_FROM_LAST_ORDER").isNotNull()) &
        (F.avg("AVG_TOTAL_TIME_FROM_LAST_ORDER").over(w_general).isNotNull()) &
        F.col("HAS_REQUIRED_HISTORY"),
        F.signum(
            F.col("AVG_TOTAL_TIME_FROM_LAST_ORDER") - F.avg("AVG_TOTAL_TIME_FROM_LAST_ORDER").over(w_general)
        ).cast("int")
    )
).withColumn(
    "SIGN_MEDIAN_FREQUENCY_CHANGE",
    F.when(
        (F.col("MEDIAN_TOTAL_TIME_FROM_LAST_ORDER").isNotNull()) &
        (F.expr("percentile_approx(MEDIAN_TOTAL_TIME_FROM_LAST_ORDER, 0.5)").over(w_general).isNotNull()) &
        F.col("HAS_REQUIRED_HISTORY"),
        F.signum(
            F.col("MEDIAN_TOTAL_TIME_FROM_LAST_ORDER") - F.expr("percentile_approx(MEDIAN_TOTAL_TIME_FROM_LAST_ORDER, 0.5)").over(w_general)
        ).cast("int")
    )
).withColumn( # ---------------- VALUE ----------------
    "SIGN_AVG_VALUE_CHANGE",
    F.when(
        (F.col("TOTAL_ORDER_VALUE_EUR").isNotNull()) &
        (F.avg("TOTAL_ORDER_VALUE_EUR").over(w_general).isNotNull()) &
        F.col("HAS_REQUIRED_HISTORY"),
        F.signum(
            F.col("TOTAL_ORDER_VALUE_EUR") - F.avg("TOTAL_ORDER_VALUE_EUR").over(w_general)
        ).cast("int")
    )
).withColumn(
    "SIGN_MEDIAN_VALUE_CHANGE",
    F.when(
        (F.col("TOTAL_ORDER_VALUE_EUR").isNotNull()) &
        (F.expr("percentile_approx(TOTAL_ORDER_VALUE_EUR, 0.5)").over(w_general).isNotNull()) &
        F.col("HAS_REQUIRED_HISTORY"),
        F.signum(
            F.col("TOTAL_ORDER_VALUE_EUR") - F.expr("percentile_approx(TOTAL_ORDER_VALUE_EUR, 0.5)").over(w_general)
        ).cast("int")
    )
).withColumn( # ---------------- QUANTITY ----------------
    "SIGN_AVG_QUANTITY_CHANGE",
    F.when(
        (F.col("TOTAL_ORDER_QUANTITY").isNotNull()) &
        (F.avg("TOTAL_ORDER_QUANTITY").over(w_general).isNotNull()) &
        F.col("HAS_REQUIRED_HISTORY"),
        F.signum(
            F.col("TOTAL_ORDER_QUANTITY") - F.avg("TOTAL_ORDER_QUANTITY").over(w_general)
        ).cast("int")
    )
).withColumn(
    "SIGN_MEDIAN_QUANTITY_CHANGE",
    F.when(
        (F.col("TOTAL_ORDER_QUANTITY").isNotNull()) &
        (F.expr("percentile_approx(TOTAL_ORDER_QUANTITY, 0.5)").over(w_general).isNotNull()) &
        F.col("HAS_REQUIRED_HISTORY"),
        F.signum(
            F.col("TOTAL_ORDER_QUANTITY") - F.expr("percentile_approx(TOTAL_ORDER_QUANTITY, 0.5)").over(w_general)
        ).cast("int")
    )
).withColumn(# ---------------- MIX SIMILARITY ----------------
    "SIGN_AVG_MIX_CHANGE",
    F.when(
        (F.col("BRAND_FORMAT_MIX_SIMILARITY").isNotNull()) &
        (F.avg("BRAND_FORMAT_MIX_SIMILARITY").over(w_general).isNotNull()) &
        F.col("HAS_REQUIRED_HISTORY"),
        F.signum(
            F.col("BRAND_FORMAT_MIX_SIMILARITY") - F.avg("BRAND_FORMAT_MIX_SIMILARITY").over(w_general)
        ).cast("int")
    )
).withColumn(
    "SIGN_MEDIAN_MIX_CHANGE",
    F.when(
        (F.col("BRAND_FORMAT_MIX_SIMILARITY").isNotNull()) &
        (F.expr("percentile_approx(BRAND_FORMAT_MIX_SIMILARITY, 0.5)").over(w_general).isNotNull()) &
        F.col("HAS_REQUIRED_HISTORY"),
        F.signum(
            F.col("BRAND_FORMAT_MIX_SIMILARITY") - F.expr("percentile_approx(BRAND_FORMAT_MIX_SIMILARITY, 0.5)").over(w_general)
        ).cast("int")
    )
)
display(df_monthly_deviations_signes.orderBy("CUSTOMER_CODE","BASELINE_DATE"))

### General calculations 

In [0]:
# ---------------- Calculations for the RULES -------------------

def median_of_sorted_array_sql(sorted_array_col: str) -> str:
  # ARRAY MUST BE SORTED ASCENDING
    return f"""
    CASE
      WHEN size({sorted_array_col}) = 0 THEN NULL
      WHEN size({sorted_array_col}) % 2 = 1
        THEN element_at({sorted_array_col}, cast(size({sorted_array_col}) / 2 as int) + 1)
      ELSE
        (
          element_at({sorted_array_col}, cast(size({sorted_array_col}) / 2 as int)) +
          element_at({sorted_array_col}, cast(size({sorted_array_col}) / 2 as int) + 1)
        ) / 2D
    END
    """

monthly_deviations_work_df = (
    df_monthly_deviations_signes
    # .withColumn("HIST_MONTHS_AVAILABLE", F.count(F.lit(1)).over(w_general))
    # .withColumn("HAS_REQUIRED_HISTORY", F.col("HIST_MONTHS_AVAILABLE") == F.lit(REQUIRED_HISTORY_LENGTH))

    # Avarage of n events in a month
    .withColumn("HISTORICAL_AVG_N_EVENTS_MONTH",F.avg("N_EVENTS_MONTH").over(w_general))
    # Calculate sum of values
    .withColumn("HISTORICAL_TOTAL_ORDER_QUANTITY",F.sum("TOTAL_ORDER_QUANTITY").over(w_general))

    # ---------- means / stddev ----------
    .withColumn("HIST_AVG_AVG_FREQUENCY", 
        F.when(F.col("HAS_REQUIRED_HISTORY"), F.avg("AVG_TOTAL_TIME_FROM_LAST_ORDER").over(w_general))
    )
    .withColumn("HIST_STD_FREQUENCY_INPUT", 
        F.when(F.col("HAS_REQUIRED_HISTORY"), F.stddev_samp("AVG_TOTAL_TIME_FROM_LAST_ORDER").over(w_general))
    )
    .withColumn("HIST_AVG_VALUE", 
        F.when(F.col("HAS_REQUIRED_HISTORY"), F.avg("TOTAL_ORDER_VALUE_EUR").over(w_general))
    )
    .withColumn("HIST_STD_VALUE", 
        F.when(
            F.col("HAS_REQUIRED_HISTORY"),
            F.when(
                F.stddev_samp("TOTAL_ORDER_VALUE_EUR").over(w_general) == 0,
                F.lit(0.001)
            ).otherwise(
                F.stddev_samp("TOTAL_ORDER_VALUE_EUR").over(w_general)
            )
        )
    )
    .withColumn("HIST_AVG_QUANTITY", 
        F.when(F.col("HAS_REQUIRED_HISTORY"), F.avg("TOTAL_ORDER_QUANTITY").over(w_general))
    )
    .withColumn("HIST_STD_QUANTITY", 
        F.when(
            F.col("HAS_REQUIRED_HISTORY"),
            F.when(
                F.stddev_samp("TOTAL_ORDER_QUANTITY").over(w_general) == 0,
                F.lit(0.001)
            ).otherwise(
                F.stddev_samp("TOTAL_ORDER_QUANTITY").over(w_general)
            )
        )
    )
    .withColumn("HIST_BRAND_FORMAT_MIX_SIMILARITY", 
        F.when(F.col("HAS_REQUIRED_HISTORY"), F.avg("BRAND_FORMAT_MIX_SIMILARITY").over(w_general))
    )
    .withColumn("HIST_STD_BRAND_FORMAT_MIX_SIMILARITY", 
        F.when(
            F.col("HAS_REQUIRED_HISTORY"),
            F.when(
                F.stddev_samp("BRAND_FORMAT_MIX_SIMILARITY").over(w_general) == 0,
                F.lit(0.001)
            ).otherwise(
                F.stddev_samp("BRAND_FORMAT_MIX_SIMILARITY").over(w_general)
            )
        )
    )

    # ---------- historical arrays ----------
    # AVG Total time from last order lists
    .withColumn("HIST_AVG_FREQ_LIST_RAW", 
        F.when(F.col("HAS_REQUIRED_HISTORY"), F.collect_list("AVG_TOTAL_TIME_FROM_LAST_ORDER").over(w_general))
    )
    .withColumn("HIST_AVG_FREQ_LIST", 
        F.when(F.col("HAS_REQUIRED_HISTORY"), F.expr("filter(HIST_AVG_FREQ_LIST_RAW, x -> x is not null)"))
    )
    .withColumn("HIST_AVG_FREQ_LIST_SORTED", 
        F.when(F.col("HAS_REQUIRED_HISTORY"), F.sort_array("HIST_AVG_FREQ_LIST"))
    )

    # MED Total time from last order lists
    .withColumn("HIST_MED_FREQ_LIST_RAW", 
        F.when(F.col("HAS_REQUIRED_HISTORY"), F.collect_list("MEDIAN_TOTAL_TIME_FROM_LAST_ORDER").over(w_general))
    )
    .withColumn("HIST_MED_FREQ_LIST", 
        F.when(F.col("HAS_REQUIRED_HISTORY"), F.expr("filter(HIST_MED_FREQ_LIST_RAW, x -> x is not null)"))
    )
    .withColumn("HIST_MED_FREQ_LIST_SORTED", 
        F.when(F.col("HAS_REQUIRED_HISTORY"), F.sort_array("HIST_MED_FREQ_LIST"))
    )

    # Value lists
    .withColumn("HIST_VALUE_LIST_RAW", 
        F.when(F.col("HAS_REQUIRED_HISTORY"), F.collect_list("TOTAL_ORDER_VALUE_EUR").over(w_general))
    )
    .withColumn("HIST_VALUE_LIST", 
        F.when(F.col("HAS_REQUIRED_HISTORY"), F.expr("filter(HIST_VALUE_LIST_RAW, x -> x is not null)"))
    )
    .withColumn("HIST_VALUE_LIST_SORTED", 
        F.when(F.col("HAS_REQUIRED_HISTORY"), F.sort_array("HIST_VALUE_LIST"))
    )

    # Quantity lists
    .withColumn("HIST_QUANTITY_LIST_RAW", 
        F.when(F.col("HAS_REQUIRED_HISTORY"), F.collect_list("TOTAL_ORDER_QUANTITY").over(w_general))
    )
    .withColumn("HIST_QUANTITY_LIST", 
        F.when(F.col("HAS_REQUIRED_HISTORY"), F.expr("filter(HIST_QUANTITY_LIST_RAW, x -> x is not null)"))
    )
    .withColumn("HIST_QUANTITY_LIST_SORTED", 
        F.when(F.col("HAS_REQUIRED_HISTORY"), F.sort_array("HIST_QUANTITY_LIST"))
    )

    # BRAND_FORMAT_MIX_SIMILARITY lists
    .withColumn("HIST_BRAND_FORMAT_MIX_SIMILARITY_LIST_RAW", 
        F.when(F.col("HAS_REQUIRED_HISTORY"), F.collect_list("BRAND_FORMAT_MIX_SIMILARITY").over(w_general))
    )
    .withColumn("HIST_BRAND_FORMAT_MIX_SIMILARITY_LIST", 
        F.when(F.col("HAS_REQUIRED_HISTORY"), F.expr("filter(HIST_BRAND_FORMAT_MIX_SIMILARITY_LIST_RAW, x -> x is not null)"))
    )
    .withColumn("HIST_BRAND_FORMAT_MIX_SIMILARITY_LIST_SORTED", 
        F.when(F.col("HAS_REQUIRED_HISTORY"), F.sort_array("HIST_BRAND_FORMAT_MIX_SIMILARITY_LIST"))
    )

    # ---------- Sign lists ----------
    .withColumn("HIST_SIGN_AVG_FREQ_LIST", 
        F.when(F.col("HAS_REQUIRED_HISTORY"), F.collect_list("SIGN_AVG_FREQUENCY_CHANGE").over(w_general))
    )
    .withColumn("HIST_SIGN_MED_FREQ_LIST", 
        F.when(F.col("HAS_REQUIRED_HISTORY"), F.collect_list("SIGN_MEDIAN_FREQUENCY_CHANGE").over(w_general))
    )
    .withColumn("HIST_SIGN_AVG_VALUE_LIST", 
        F.when(F.col("HAS_REQUIRED_HISTORY"), F.collect_list("SIGN_AVG_VALUE_CHANGE").over(w_general))
    )
    .withColumn("HIST_SIGN_MED_VALUE_LIST", 
        F.when(F.col("HAS_REQUIRED_HISTORY"), F.collect_list("SIGN_MEDIAN_VALUE_CHANGE").over(w_general))
    )
    .withColumn("HIST_SIGN_AVG_QUANTITY_LIST", 
        F.when(F.col("HAS_REQUIRED_HISTORY"), F.collect_list("SIGN_AVG_QUANTITY_CHANGE").over(w_general))
    )
    .withColumn("HIST_SIGN_MED_QUANTITY_LIST", 
        F.when(F.col("HAS_REQUIRED_HISTORY"), F.collect_list("SIGN_MEDIAN_QUANTITY_CHANGE").over(w_general))
    )
    .withColumn("HIST_SIGN_AVG_MIX_LIST", 
        F.when(F.col("HAS_REQUIRED_HISTORY"), F.collect_list("SIGN_AVG_MIX_CHANGE").over(w_general))
    )
    .withColumn("HIST_SIGN_MED_MIX_LIST", 
        F.when(F.col("HAS_REQUIRED_HISTORY"), F.collect_list("SIGN_MEDIAN_MIX_CHANGE").over(w_general))
    )

    # ---------- medians from sorted arrays ----------
    .withColumn("HIST_MEDIAN_FREQUENCY", 
        F.when(F.col("HAS_REQUIRED_HISTORY"), F.expr(median_of_sorted_array_sql("HIST_MED_FREQ_LIST_SORTED")))
    )
    .withColumn("HIST_MEDIAN_FREQUENCY_AVG", 
        F.when(F.col("HAS_REQUIRED_HISTORY"), F.expr(median_of_sorted_array_sql("HIST_AVG_FREQ_LIST_SORTED")))
    )
    .withColumn("HIST_MEDIAN_VALUE", 
        F.when(F.col("HAS_REQUIRED_HISTORY"), F.expr(median_of_sorted_array_sql("HIST_VALUE_LIST_SORTED")))
    )
    .withColumn("HIST_MEDIAN_QUANTITY", 
        F.when(F.col("HAS_REQUIRED_HISTORY"), F.expr(median_of_sorted_array_sql("HIST_QUANTITY_LIST_SORTED")))
    )
    .withColumn("HIST_MEDIAN_BRAND_FORMAT_MIX_SIMILARITY", 
        F.when(F.col("HAS_REQUIRED_HISTORY"), F.expr(median_of_sorted_array_sql("HIST_BRAND_FORMAT_MIX_SIMILARITY_LIST_SORTED")))
    )

    # ---------- MAD arrays ----------
    .withColumn(
        "HIST_FREQ_ABS_DEV_LIST",
        F.when(F.col("HAS_REQUIRED_HISTORY"), F.expr("transform(HIST_MED_FREQ_LIST, x -> abs(x - HIST_MEDIAN_FREQUENCY))"))
    )
    .withColumn(
        "HIST_FREQ_AVG_ABS_DEV_LIST",
        F.when(F.col("HAS_REQUIRED_HISTORY"), F.expr("transform(HIST_AVG_FREQ_LIST, x -> abs(x - HIST_MEDIAN_FREQUENCY_AVG))"))
    )
    .withColumn(
        "HIST_VALUE_ABS_DEV_LIST",
        F.when(F.col("HAS_REQUIRED_HISTORY"), F.expr("transform(HIST_VALUE_LIST, x -> abs(x - HIST_MEDIAN_VALUE))"))
    )
    .withColumn(
        "HIST_QUANTITY_ABS_DEV_LIST",
        F.when(F.col("HAS_REQUIRED_HISTORY"), F.expr("transform(HIST_QUANTITY_LIST, x -> abs(x - HIST_MEDIAN_QUANTITY))"))
    )
    .withColumn(
        "HIST_BRAND_FORMAT_MIX_SIMILARITY_ABS_DEV_LIST",
        F.when(F.col("HAS_REQUIRED_HISTORY"), F.expr("transform(HIST_BRAND_FORMAT_MIX_SIMILARITY_LIST, x -> abs(x - HIST_MEDIAN_BRAND_FORMAT_MIX_SIMILARITY))"))
    )


    .withColumn("HIST_FREQ_ABS_DEV_LIST_SORTED", 
        F.when(F.col("HAS_REQUIRED_HISTORY"), F.sort_array("HIST_FREQ_ABS_DEV_LIST"))
    )
    .withColumn("HIST_FREQ_AVG_ABS_DEV_LIST_SORTED", 
        F.when(F.col("HAS_REQUIRED_HISTORY"), F.sort_array("HIST_FREQ_AVG_ABS_DEV_LIST"))
    )
    .withColumn("HIST_VALUE_ABS_DEV_LIST_SORTED", 
        F.when(F.col("HAS_REQUIRED_HISTORY"), F.sort_array("HIST_VALUE_ABS_DEV_LIST"))
    )
    .withColumn("HIST_QUANTITY_ABS_DEV_LIST_SORTED", 
        F.when(F.col("HAS_REQUIRED_HISTORY"), F.sort_array("HIST_QUANTITY_ABS_DEV_LIST"))
    )
    .withColumn("HIST_BRAND_FORMAT_MIX_SIMILARITY_ABS_DEV_LIST_SORTED", 
        F.when(F.col("HAS_REQUIRED_HISTORY"), F.sort_array("HIST_BRAND_FORMAT_MIX_SIMILARITY_ABS_DEV_LIST"))
    )


    # .withColumn("HIST_MAD_FREQUENCY", 
    #     F.when(F.col("HAS_REQUIRED_HISTORY"), F.expr(median_of_sorted_array_sql("HIST_FREQ_ABS_DEV_LIST_SORTED")))
    # )
    .withColumn(
        "HIST_MAD_FREQUENCY",
        F.when(
            F.col("HAS_REQUIRED_HISTORY"),
            F.when(
                F.expr(median_of_sorted_array_sql("HIST_FREQ_ABS_DEV_LIST_SORTED")) == 0,
                F.lit(0.001)
            ).otherwise(
                F.expr(median_of_sorted_array_sql("HIST_FREQ_ABS_DEV_LIST_SORTED"))
            )
        )
    )
    .withColumn(
        "HIST_MAD_FREQUENCY_AVG",
        F.when(
            F.col("HAS_REQUIRED_HISTORY"),
            F.when(
                F.expr(median_of_sorted_array_sql("HIST_FREQ_AVG_ABS_DEV_LIST_SORTED")) == 0,
                F.lit(0.001)
            ).otherwise(
                F.expr(median_of_sorted_array_sql("HIST_FREQ_AVG_ABS_DEV_LIST_SORTED"))
            )
        )
    )
    .withColumn(
        "HIST_MAD_VALUE",
        F.when(
            F.col("HAS_REQUIRED_HISTORY"),
            F.when(
                F.expr(median_of_sorted_array_sql("HIST_VALUE_ABS_DEV_LIST_SORTED")) == 0,
                F.lit(0.001)
            ).otherwise(
                F.expr(median_of_sorted_array_sql("HIST_VALUE_ABS_DEV_LIST_SORTED"))
            )
        )
    )
    .withColumn(
        "HIST_MAD_QUANTITY",
        F.when(
            F.col("HAS_REQUIRED_HISTORY"),
            F.when(
                F.expr(median_of_sorted_array_sql("HIST_QUANTITY_ABS_DEV_LIST_SORTED")) == 0,
                F.lit(0.001)
            ).otherwise(
                F.expr(median_of_sorted_array_sql("HIST_QUANTITY_ABS_DEV_LIST_SORTED"))
            )
        )
    )
    .withColumn(
        "HIST_MAD_BRAND_FORMAT_MIX_SIMILARITY",
        F.when(
            F.col("HAS_REQUIRED_HISTORY"),
            F.when(
                F.expr(median_of_sorted_array_sql("HIST_BRAND_FORMAT_MIX_SIMILARITY_ABS_DEV_LIST_SORTED")) == 0,
                F.lit(0.001)
            ).otherwise(
                F.expr(median_of_sorted_array_sql("HIST_BRAND_FORMAT_MIX_SIMILARITY_ABS_DEV_LIST_SORTED"))
            )
        )
    )

    #------------------ 2sd deviation calculations (RULE 3) ---------------------
    # Flag every row based on its OWN history for each higher and lower bound
    .withColumn(
        "IS_UPPER_OUTLIER_FREQUENCY",
        F.when(
            (F.col("HAS_REQUIRED_HISTORY")) & 
            (F.col("AVG_TOTAL_TIME_FROM_LAST_ORDER") > (F.col("HIST_AVG_AVG_FREQUENCY") + 2 * F.col("HIST_STD_FREQUENCY_INPUT"))),
            1
        ).otherwise(0)
    )
    .withColumn(
        "IS_LOWER_OUTLIER_FREQUENCY",
        F.when(
            (F.col("HAS_REQUIRED_HISTORY")) &
            (F.col("AVG_TOTAL_TIME_FROM_LAST_ORDER") < (F.col("HIST_AVG_AVG_FREQUENCY") - 2 * F.col("HIST_STD_FREQUENCY_INPUT"))),
            1
        ).otherwise(0)
    )
    .withColumn(
        "IS_UPPER_OUTLIER_MEDIAN_FREQUENCY",
        F.when(
            (F.col("HAS_REQUIRED_HISTORY")) &
            (F.col("MEDIAN_TOTAL_TIME_FROM_LAST_ORDER") > (F.col("HIST_MEDIAN_FREQUENCY") + 2 * F.col("HIST_MAD_FREQUENCY"))),
            1
        ).otherwise(0)
    )
    .withColumn(
        "IS_LOWER_OUTLIER_MEDIAN_FREQUENCY",
        F.when(
            (F.col("HAS_REQUIRED_HISTORY")) &
            (F.col("MEDIAN_TOTAL_TIME_FROM_LAST_ORDER") < (F.col("HIST_MEDIAN_FREQUENCY") - 2 * F.col("HIST_MAD_FREQUENCY"))),
            1
        ).otherwise(0)
    )
    .withColumn(
        "IS_UPPER_OUTLIER_VALUE",
        F.when(
            (F.col("HAS_REQUIRED_HISTORY")) &
            (F.col("TOTAL_ORDER_VALUE_EUR") > (F.col("HIST_AVG_VALUE") + 2 * F.col("HIST_STD_VALUE"))),
            1
       ).otherwise(0)
    )
    .withColumn(
        "IS_LOWER_OUTLIER_VALUE",
        F.when(
            (F.col("HAS_REQUIRED_HISTORY")) &
            (F.col("TOTAL_ORDER_VALUE_EUR") < (F.col("HIST_AVG_VALUE") - 2 * F.col("HIST_STD_VALUE"))),
            1
       ).otherwise(0)
    )
    .withColumn(
        "IS_UPPER_OUTLIER_MEDIAN_VALUE",
        F.when(
            (F.col("HAS_REQUIRED_HISTORY")) &
            (F.col("TOTAL_ORDER_VALUE_EUR") > (F.col("HIST_MEDIAN_VALUE") + 2 * F.col("HIST_MAD_VALUE"))),
            1
       ).otherwise(0)
    )
    .withColumn(
        "IS_LOWER_OUTLIER_MEDIAN_VALUE",
        F.when(
            (F.col("HAS_REQUIRED_HISTORY")) &
            (F.col("TOTAL_ORDER_VALUE_EUR") < (F.col("HIST_MEDIAN_VALUE") - 2 * F.col("HIST_MAD_VALUE"))),
            1
       ).otherwise(0)
    )
    .withColumn(
        "IS_UPPER_OUTLIER_QUANTITY",
        F.when(
            (F.col("HAS_REQUIRED_HISTORY")) &
            (F.col("TOTAL_ORDER_QUANTITY") > (F.col("HIST_AVG_QUANTITY") + 2 * F.col("HIST_STD_QUANTITY"))),
            1
        ).otherwise(0)
    )
    .withColumn(
        "IS_LOWER_OUTLIER_QUANTITY",
        F.when(
            (F.col("HAS_REQUIRED_HISTORY")) &
            (F.col("TOTAL_ORDER_QUANTITY") < (F.col("HIST_AVG_QUANTITY") - 2 * F.col("HIST_STD_QUANTITY"))),
            1
        ).otherwise(0)
    )
    .withColumn(
        "IS_UPPER_OUTLIER_MEDIAN_QUANTITY",
        F.when(
            (F.col("HAS_REQUIRED_HISTORY")) &
            (F.col("TOTAL_ORDER_QUANTITY") > (F.col("HIST_MEDIAN_QUANTITY") + 2 * F.col("HIST_MAD_QUANTITY"))),
            1
        ).otherwise(0)
    )
    .withColumn(
        "IS_LOWER_OUTLIER_MEDIAN_QUANTITY",
        F.when(
            (F.col("HAS_REQUIRED_HISTORY")) &
            (F.col("TOTAL_ORDER_QUANTITY") < (F.col("HIST_MEDIAN_QUANTITY") - 2 * F.col("HIST_MAD_QUANTITY"))),
            1
        ).otherwise(0)
    )
    .withColumn(
        "IS_UPPER_OUTLIER_MIX",
        F.when(
            (F.col("HAS_REQUIRED_HISTORY")) &
            (F.col("BRAND_FORMAT_MIX_SIMILARITY") > (F.col("HIST_BRAND_FORMAT_MIX_SIMILARITY") + 2 * F.col("HIST_STD_BRAND_FORMAT_MIX_SIMILARITY"))),
            1
        ).otherwise(0)
    )
    .withColumn(
        "IS_LOWER_OUTLIER_MIX",
        F.when(
            (F.col("HAS_REQUIRED_HISTORY")) &
            (F.col("BRAND_FORMAT_MIX_SIMILARITY") < (F.col("HIST_BRAND_FORMAT_MIX_SIMILARITY") - 2 * F.col("HIST_STD_BRAND_FORMAT_MIX_SIMILARITY"))),
            1
        ).otherwise(0)
    )
    .withColumn(
        "IS_UPPER_OUTLIER_MEDIAN_MIX",
        F.when(
            (F.col("HAS_REQUIRED_HISTORY")) &
            (F.col("BRAND_FORMAT_MIX_SIMILARITY") > (F.col("HIST_MEDIAN_BRAND_FORMAT_MIX_SIMILARITY") + 2 * F.col("HIST_MAD_BRAND_FORMAT_MIX_SIMILARITY"))),
            1
        ).otherwise(0)
    )
    .withColumn(
        "IS_LOWER_OUTLIER_MEDIAN_MIX",
        F.when(
            (F.col("HAS_REQUIRED_HISTORY")) &
            (F.col("BRAND_FORMAT_MIX_SIMILARITY") < (F.col("HIST_MEDIAN_BRAND_FORMAT_MIX_SIMILARITY") - 2 * F.col("HIST_MAD_BRAND_FORMAT_MIX_SIMILARITY"))),
            1
        ).otherwise(0)
    )
    
    # Sum up the highr and lower bounds for each rows history 
    .withColumn("PAST_UPPER_OUTLIERS_COUNT_FREQUENCY", F.sum("IS_UPPER_OUTLIER_FREQUENCY").over(w_general)) \
    .withColumn("PAST_LOWER_OUTLIERS_COUNT_FREQUENCY", F.sum("IS_LOWER_OUTLIER_FREQUENCY").over(w_general)) \
    .withColumn("PAST_UPPER_OUTLIERS_COUNT_MEDIAN_FREQUENCY", F.sum("IS_UPPER_OUTLIER_MEDIAN_FREQUENCY").over(w_general)) \
    .withColumn("PAST_LOWER_OUTLIERS_COUNT_MEDIAN_FREQUENCY", F.sum("IS_LOWER_OUTLIER_MEDIAN_FREQUENCY").over(w_general)) \
    .withColumn("PAST_UPPER_OUTLIERS_COUNT_VALUE", F.sum("IS_UPPER_OUTLIER_VALUE").over(w_general)) \
    .withColumn("PAST_LOWER_OUTLIERS_COUNT_VALUE", F.sum("IS_LOWER_OUTLIER_VALUE").over(w_general)) \
    .withColumn("PAST_UPPER_OUTLIERS_COUNT_MEDIAN_VALUE", F.sum("IS_UPPER_OUTLIER_MEDIAN_VALUE").over(w_general)) \
    .withColumn("PAST_LOWER_OUTLIERS_COUNT_MEDIAN_VALUE", F.sum("IS_LOWER_OUTLIER_MEDIAN_VALUE").over(w_general)) \
    .withColumn("PAST_UPPER_OUTLIERS_COUNT_QUANTITY", F.sum("IS_UPPER_OUTLIER_QUANTITY").over(w_general)) \
    .withColumn("PAST_LOWER_OUTLIERS_COUNT_QUANTITY", F.sum("IS_LOWER_OUTLIER_QUANTITY").over(w_general)) \
    .withColumn("PAST_UPPER_OUTLIERS_COUNT_MEDIAN_QUANTITY", F.sum("IS_UPPER_OUTLIER_MEDIAN_QUANTITY").over(w_general)) \
    .withColumn("PAST_LOWER_OUTLIERS_COUNT_MEDIAN_QUANTITY", F.sum("IS_LOWER_OUTLIER_MEDIAN_QUANTITY").over(w_general)) \
    .withColumn("PAST_UPPER_OUTLIERS_COUNT_MIX", F.sum("IS_UPPER_OUTLIER_MIX").over(w_general)) \
    .withColumn("PAST_LOWER_OUTLIERS_COUNT_MIX", F.sum("IS_LOWER_OUTLIER_MIX").over(w_general)) \
    .withColumn("PAST_UPPER_OUTLIERS_COUNT_MEDIAN_MIX", F.sum("IS_UPPER_OUTLIER_MEDIAN_MIX").over(w_general)) \
    .withColumn("PAST_LOWER_OUTLIERS_COUNT_MEDIAN_MIX", F.sum("IS_LOWER_OUTLIER_MEDIAN_MIX").over(w_general))
)
# Ensure AGGREGATE_BY is present in output
monthly_deviations_work_df = monthly_deviations_work_df.select("*", "AGGREGATE_BY")

#display(monthly_deviations_work_df)

In [0]:
#################### Final MONTHLY_aggregated_table ###############
MONTHLY_aggregated_table_df = monthly_deviations_work_df.select(
    "EVENT_IDENTIFIER",
    "AGGREGATE_BY",
    "CUSTOMER_CODE",
    "BASELINE_DATE",
    "N_EVENTS_MONTH",
    "AVG_TOTAL_TIME_FROM_LAST_ORDER",
    "MEDIAN_TOTAL_TIME_FROM_LAST_ORDER",
    "TOTAL_ORDER_VALUE_EUR",
    "TOTAL_ORDER_QUANTITY",
    "TOTAL_BRAND_FORMAT_MIX",
    "FRACTION_BRAND_FORMAT_MIX",
    "HISTORICAL_AVG_N_EVENTS_MONTH",
    F.col("HIST_AVG_AVG_FREQUENCY").alias("HISTORICAL_AVG_TIME_FROM_LAST_ORDER​"),
    F.col("HIST_AVG_VALUE").alias("HISTORICAL_AVG_TOTAL_ORDER_VALUE_EUR ​"),
    "HISTORICAL_TOTAL_ORDER_QUANTITY",
    F.col("HIST_AVG_QUANTITY").alias("HISTORICAL_AVG_TOTAL_ORDER_QUANTITY​"),
    "HISTORICAL_TOTAL_BRAND_FORMAT_MIX",
    "HISTORICAL_FRACTION_BRAND_FORMAT_MIX",
    "HIST_EVENT_COUNT"

)
# if write: 
#     agg_df_final_month.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(monthly_aggregated_table)
display(MONTHLY_aggregated_table_df) 

## MONTHLY_DEVIATIONS_TABLE

## RULE 1

In [0]:
# ------------------- RULE 1 -----------------------
df_rule_1 = monthly_deviations_work_df \
    .withColumn("STDEV_FREQUENCY", F.try_divide(F.col("AVG_TOTAL_TIME_FROM_LAST_ORDER") - F.col("HIST_AVG_AVG_FREQUENCY"), F.col("HIST_STD_FREQUENCY_INPUT")))\
    .withColumn("MAD_FREQUENCY", F.try_divide((F.col("MEDIAN_TOTAL_TIME_FROM_LAST_ORDER") - F.col("HIST_MEDIAN_FREQUENCY")), F.col("HIST_MAD_FREQUENCY")))\
    .withColumn("MAD_FREQUENCY_ON_AVG", F.try_divide((F.col("AVG_TOTAL_TIME_FROM_LAST_ORDER") - F.col("HIST_MEDIAN_FREQUENCY_AVG")), F.col("HIST_MAD_FREQUENCY_AVG")))\
    .withColumn("STDEV_VALUE", F.try_divide(F.col("TOTAL_ORDER_VALUE_EUR") - F.col("HIST_AVG_VALUE"), F.col("HIST_STD_VALUE")))\
    .withColumn("MAD_VALUE", F.try_divide((F.col("TOTAL_ORDER_VALUE_EUR") - F.col("HIST_MEDIAN_VALUE")), F.col("HIST_MAD_VALUE")))\
    .withColumn("STDEV_QUANTITY", F.try_divide(F.col("TOTAL_ORDER_QUANTITY") - F.col("HIST_AVG_QUANTITY"), F.col("HIST_STD_QUANTITY")))\
    .withColumn("MAD_QUANTITY", F.try_divide((F.col("TOTAL_ORDER_QUANTITY") - F.col("HIST_MEDIAN_QUANTITY")), F.col("HIST_MAD_QUANTITY")))\
    .withColumn("STDEV_MIX", F.try_divide(F.col("BRAND_FORMAT_MIX_SIMILARITY") - F.col("HIST_BRAND_FORMAT_MIX_SIMILARITY"), F.col("HIST_STD_BRAND_FORMAT_MIX_SIMILARITY")))\
    .withColumn("MAD_MIX", F.try_divide((F.col("BRAND_FORMAT_MIX_SIMILARITY") - F.col("HIST_MEDIAN_BRAND_FORMAT_MIX_SIMILARITY")), F.col("HIST_MAD_BRAND_FORMAT_MIX_SIMILARITY")))\

# display(df_rule_1)

## RULE 2

In [0]:
# Calculate the number of consecutive history events with the same sign of deviation as the current event
def consecutive_same_sign_count(array_col, sign_col):
    return F.expr(f"""
        aggregate(
            reverse(slice(reverse({array_col}), 1, {REQUIRED_HISTORY_LENGTH})),
            0,
            (acc, x) -> IF(x = {sign_col} AND x IS NOT NULL, acc + 1, 0)
        )
    """)

df_rule_2 = (
    df_rule_1
    .withColumn("NUM_SAME_DIRECTION_AVG_FREQUENCY", consecutive_same_sign_count("HIST_SIGN_AVG_FREQ_LIST", "SIGN_AVG_FREQUENCY_CHANGE"))
    .withColumn("NUM_SAME_DIRECTION_MEDIAN_FREQUENCY", consecutive_same_sign_count("HIST_SIGN_MED_FREQ_LIST", "SIGN_MEDIAN_FREQUENCY_CHANGE"))
    .withColumn("NUM_SAME_DIRECTION_AVG_VALUE", consecutive_same_sign_count("HIST_SIGN_AVG_VALUE_LIST", "SIGN_AVG_VALUE_CHANGE"))
    .withColumn("NUM_SAME_DIRECTION_MEDIAN_VALUE", consecutive_same_sign_count("HIST_SIGN_MED_VALUE_LIST", "SIGN_MEDIAN_VALUE_CHANGE"))
    .withColumn("NUM_SAME_DIRECTION_AVG_QUANTITY", consecutive_same_sign_count("HIST_SIGN_AVG_QUANTITY_LIST", "SIGN_AVG_QUANTITY_CHANGE"))
    .withColumn("NUM_SAME_DIRECTION_MEDIAN_QUANTITY", consecutive_same_sign_count("HIST_SIGN_MED_QUANTITY_LIST", "SIGN_MEDIAN_QUANTITY_CHANGE"))
    .withColumn("NUM_SAME_DIRECTION_AVG_MIX", consecutive_same_sign_count("HIST_SIGN_AVG_MIX_LIST", "SIGN_AVG_MIX_CHANGE"))
    .withColumn("NUM_SAME_DIRECTION_MEDIAN_MIX", consecutive_same_sign_count("HIST_SIGN_MED_MIX_LIST", "SIGN_MEDIAN_MIX_CHANGE"))
)

#display(df_rule_2)

## RULE 3

In [0]:

# If current is upper outlier, return past upper count. If lower, return past lower count.
df_rule_3 = df_rule_2.withColumn("NUM_SAME_2SD_DIRECTION_AVG_FREQUENCY", 
    F.when(~F.col("HAS_REQUIRED_HISTORY"), F.lit(None)).when(F.col("IS_UPPER_OUTLIER_FREQUENCY") == 1, F.col("PAST_UPPER_OUTLIERS_COUNT_FREQUENCY"))
     .when(F.col("IS_LOWER_OUTLIER_FREQUENCY") == 1, F.col("PAST_LOWER_OUTLIERS_COUNT_FREQUENCY") )
     .otherwise(0)
).withColumn("NUM_SAME_2SD_DIRECTION_MEDIAN_FREQUENCY",
    F.when(~F.col("HAS_REQUIRED_HISTORY"), F.lit(None)).when(F.col("IS_UPPER_OUTLIER_MEDIAN_FREQUENCY") == 1, F.col("PAST_UPPER_OUTLIERS_COUNT_MEDIAN_FREQUENCY"))
     .when(F.col("IS_LOWER_OUTLIER_MEDIAN_FREQUENCY") == 1, F.col("PAST_LOWER_OUTLIERS_COUNT_MEDIAN_FREQUENCY"))
     .otherwise(0)
).withColumn("NUM_SAME_2SD_DIRECTION_AVG_VALUE",
    F.when(~F.col("HAS_REQUIRED_HISTORY"), F.lit(None)).when(F.col("IS_UPPER_OUTLIER_VALUE") == 1, F.col("PAST_UPPER_OUTLIERS_COUNT_VALUE"))
     .when(F.col("IS_LOWER_OUTLIER_VALUE") == 1, F.col("PAST_LOWER_OUTLIERS_COUNT_VALUE"))
     .otherwise(0)
).withColumn("NUM_SAME_2SD_DIRECTION_MEDIAN_VALUE",
    F.when(~F.col("HAS_REQUIRED_HISTORY"), F.lit(None)).when(F.col("IS_UPPER_OUTLIER_MEDIAN_VALUE") == 1, F.col("PAST_UPPER_OUTLIERS_COUNT_MEDIAN_VALUE"))
     .when(F.col("IS_LOWER_OUTLIER_MEDIAN_VALUE") == 1, F.col("PAST_LOWER_OUTLIERS_COUNT_MEDIAN_VALUE"))
     .otherwise(0)
).withColumn("NUM_SAME_2SD_DIRECTION_AVG_QUANTITY",
    F.when(~F.col("HAS_REQUIRED_HISTORY"), F.lit(None)).when(F.col("IS_UPPER_OUTLIER_QUANTITY") == 1, F.col("PAST_UPPER_OUTLIERS_COUNT_QUANTITY"))
     .when(F.col("IS_LOWER_OUTLIER_QUANTITY") == 1, F.col("PAST_LOWER_OUTLIERS_COUNT_QUANTITY"))
     .otherwise(0)
).withColumn("NUM_SAME_2SD_DIRECTION_MEDIAN_QUANTITY",
    F.when(~F.col("HAS_REQUIRED_HISTORY"), F.lit(None)).when(F.col("IS_UPPER_OUTLIER_MEDIAN_QUANTITY") == 1, F.col("PAST_UPPER_OUTLIERS_COUNT_MEDIAN_QUANTITY"))
     .when(F.col("IS_LOWER_OUTLIER_MEDIAN_QUANTITY") == 1, F.col("PAST_LOWER_OUTLIERS_COUNT_MEDIAN_QUANTITY"))
     .otherwise(0)
).withColumn("NUM_SAME_2SD_DIRECTION_AVG_MIX",
    F.when(~F.col("HAS_REQUIRED_HISTORY"), F.lit(None)).when(F.col("IS_UPPER_OUTLIER_MIX") == 1, F.col("PAST_UPPER_OUTLIERS_COUNT_MIX"))
     .when(F.col("IS_LOWER_OUTLIER_MIX") == 1, F.col("PAST_LOWER_OUTLIERS_COUNT_MIX"))
     .otherwise(0)
).withColumn("NUM_SAME_2SD_DIRECTION_MEDIAN_MIX",
    F.when(~F.col("HAS_REQUIRED_HISTORY"), F.lit(None)).when(F.col("IS_UPPER_OUTLIER_MEDIAN_MIX") == 1, F.col("PAST_UPPER_OUTLIERS_COUNT_MEDIAN_MIX"))
     .when(F.col("IS_LOWER_OUTLIER_MEDIAN_MIX") == 1, F.col("PAST_LOWER_OUTLIERS_COUNT_MEDIAN_MIX"))
     .otherwise(0)
)
#display(df_rule_3)

## RULE 4

In [0]:
# .withColumn("PCT_AVG_FREQUENCY",F.try_divide(F.col("AVG_TOTAL_TIME_FROM_LAST_ORDER"),F.col("HIST_AVG_AVG_FREQUENCY")))\
df_rule_4 = df_rule_3\
    .withColumn(
        "PCT_AVG_FREQUENCY",
        F.when(F.col("HIST_AVG_AVG_FREQUENCY").isNull(), None)
        .when(F.col("HIST_AVG_AVG_FREQUENCY") == 0, 100)
        .otherwise(
           ((F.col("AVG_TOTAL_TIME_FROM_LAST_ORDER") - F.col("HIST_AVG_AVG_FREQUENCY")) /
                  F.col("HIST_AVG_AVG_FREQUENCY")) * 100
        )
    )\
    .withColumn(
        "PCT_MEDIAN_FREQUENCY",
        F.when(F.col("HIST_MEDIAN_FREQUENCY").isNull(), None)
        .when(F.col("HIST_MEDIAN_FREQUENCY") == 0, 100)
        .otherwise(
           ((F.col("MEDIAN_TOTAL_TIME_FROM_LAST_ORDER") - F.col("HIST_MEDIAN_FREQUENCY")) /
                  F.col("HIST_MEDIAN_FREQUENCY")) * 100
        )
    )\
    .withColumn(
        "PCT_AVG_VALUE",
        F.when(F.col("HIST_AVG_VALUE").isNull(), None)
        .when(F.col("HIST_AVG_VALUE") == 0, 100)
        .otherwise(
           ((F.col("TOTAL_ORDER_VALUE_EUR") - F.col("HIST_AVG_VALUE")) /
                  F.col("HIST_AVG_VALUE")) * 100
        )
    )\
    .withColumn(
        "PCT_MEDIAN_VALUE",
        F.when(F.col("HIST_MEDIAN_VALUE").isNull(), None)
        .when(F.col("HIST_MEDIAN_VALUE") == 0, 100)
        .otherwise(
           ((F.col("TOTAL_ORDER_VALUE_EUR") - F.col("HIST_MEDIAN_VALUE")) /
                  F.col("HIST_MEDIAN_VALUE")) * 100
        )
    )\
    .withColumn(
        "PCT_AVG_QUANTITY",
        F.when(F.col("HIST_AVG_QUANTITY").isNull(), None)
        .when(F.col("HIST_AVG_QUANTITY") == 0, 100)
        .otherwise(
           ((F.col("TOTAL_ORDER_QUANTITY") - F.col("HIST_AVG_QUANTITY")) /
                  F.col("HIST_AVG_QUANTITY")) * 100
        )
    )\
    .withColumn(
        "PCT_MEDIAN_QUANTITY",
        F.when(F.col("HIST_MEDIAN_QUANTITY").isNull(), None)
        .when(F.col("HIST_MEDIAN_QUANTITY") == 0, 100)
        .otherwise(
           ((F.col("TOTAL_ORDER_QUANTITY") - F.col("HIST_MEDIAN_QUANTITY")) /
                  F.col("HIST_MEDIAN_QUANTITY")) * 100
        )
    )\
    .withColumn(
        "PCT_AVG_MIX",
        F.when(F.col("HIST_BRAND_FORMAT_MIX_SIMILARITY").isNull(), None)
        .when(F.col("HIST_BRAND_FORMAT_MIX_SIMILARITY") == 0, 100)
        .otherwise(
           ((F.col("BRAND_FORMAT_MIX_SIMILARITY") - F.col("HIST_BRAND_FORMAT_MIX_SIMILARITY")) /
                  F.col("HIST_BRAND_FORMAT_MIX_SIMILARITY")) * 100
        )
    )\
    .withColumn(
        "PCT_MEDIAN_MIX",
        F.when(F.col("HIST_MEDIAN_BRAND_FORMAT_MIX_SIMILARITY").isNull(), None)
        .when(F.col("HIST_MEDIAN_BRAND_FORMAT_MIX_SIMILARITY") == 0, 100)
        .otherwise(
           ((F.col("BRAND_FORMAT_MIX_SIMILARITY") - F.col("HIST_MEDIAN_BRAND_FORMAT_MIX_SIMILARITY")) /
                  F.col("HIST_MEDIAN_BRAND_FORMAT_MIX_SIMILARITY")) * 100
        )
    )

# Gives 100 if it is devided by 0 and null if devided by null(no minimum required)

display(df_rule_4)

## Rule 5

In [0]:
# Calculate the percentile score with respect to the previasly sorted array
def get_percentile(target_col, array_col):
    return F.expr(f"""
        ROUND(
            try_divide(
                aggregate(
                    {array_col}, 
                    0D, 
                    (acc, x) -> acc + IF(x <= {target_col}, 1D, 0D)
                ), 
                size({array_col})
            ) * 100, 
        0)
    """)

df_rule_5 = df_rule_4\
    .withColumn("PCTILE_AVG_FREQUENCY", get_percentile("AVG_TOTAL_TIME_FROM_LAST_ORDER", "HIST_AVG_FREQ_LIST_SORTED"))\
    .withColumn("PCTILE_MEDIAN_FREQUENCY", get_percentile("MEDIAN_TOTAL_TIME_FROM_LAST_ORDER", "HIST_MED_FREQ_LIST_SORTED"))\
    .withColumn("PCTILE_VALUE",get_percentile("TOTAL_ORDER_VALUE_EUR","HIST_VALUE_LIST_SORTED"))\
    .withColumn("PCTILE_QUANTITY",get_percentile("TOTAL_ORDER_QUANTITY","HIST_QUANTITY_LIST_SORTED"))\
    .withColumn("PCTILE_AVG_MIX",get_percentile("BRAND_FORMAT_MIX_SIMILARITY","HIST_BRAND_FORMAT_MIX_SIMILARITY_LIST_SORTED"))
display(df_rule_5)

In [0]:
############## Select and Display Final MONTHLY_DEVIATIONS_TABLE ##############
df_final_monthly_deviation_table = df_rule_5.select(
    F.concat_ws(
        "_",
        F.col("CUSTOMER_CODE"),
        F.when(F.col("AGGREGATE_BY").isNull(), "None").otherwise(F.col("AGGREGATE_BY")),
        F.month("BASELINE_DATE"),
        F.year("BASELINE_DATE")
    ).alias("EVENT_IDENTIFIER"),    # Creating a new unique EVENT_IDENTIFIER
    "AGGREGATE_BY",
    "CUSTOMER_CODE",
    "BASELINE_DATE",
    "AVG_TOTAL_TIME_FROM_LAST_ORDER",
    "MEDIAN_TOTAL_TIME_FROM_LAST_ORDER",
    "TOTAL_ORDER_VALUE_EUR",
    "TOTAL_ORDER_QUANTITY",
    "FRACTION_BRAND_FORMAT_MIX",
    "REQUIRED_HISTORY_LENGTH",
    "MINIMAL_REQUIRED_HISTORY_LENGTH",
    "N_EVENTS_MONTH",
    "HISTORICAL_FRACTION_BRAND_FORMAT_MIX",
    "BRAND_FORMAT_MIX_SIMILARITY",
    "SIGN_AVG_FREQUENCY_CHANGE",
    "SIGN_MEDIAN_FREQUENCY_CHANGE",
    "SIGN_AVG_VALUE_CHANGE",
    "SIGN_MEDIAN_VALUE_CHANGE",
    "SIGN_AVG_QUANTITY_CHANGE",
    "SIGN_MEDIAN_QUANTITY_CHANGE",
    "SIGN_AVG_MIX_CHANGE",
    "SIGN_MEDIAN_MIX_CHANGE",
    "HISTORICAL_AVG_N_EVENTS_MONTH",
    F.col("HIST_AVG_AVG_FREQUENCY").alias("HISTORICAL_AVG_TIME_FROM_LAST_ORDER"),
    F.col("HIST_MEDIAN_FREQUENCY").alias("HISTORICAL_MEDIAN_TIME_FROM_LAST_ORDER"),
    F.col("HIST_AVG_VALUE").alias("HISTORICAL_AVG_TOTAL_ORDER_VALUE_EUR"),
    F.col("HIST_MEDIAN_VALUE").alias("HISTORICAL_MEDIAN_TOTAL_ORDER_VALUE_EUR"),
    "HISTORICAL_TOTAL_ORDER_QUANTITY",
    F.col("HIST_AVG_QUANTITY").alias("HISTORICAL_AVG_TOTAL_ORDER_QUANTITY"),
    F.col("HIST_MEDIAN_QUANTITY").alias("HISTORICAL_MEDIAN_TOTAL_ORDER_QUANTITY"),
    "HISTORICAL_TOTAL_BRAND_FORMAT_MIX",
    F.col("HIST_BRAND_FORMAT_MIX_SIMILARITY").alias("HISTORICAL_AVG_TOTAL_BRAND_FORMAT_MIX"),
    F.col("HIST_MEDIAN_BRAND_FORMAT_MIX_SIMILARITY").alias("HISTORICAL_MEDIAN_TOTAL_BRAND_FORMAT_MIX"),
    F.col("HIST_EVENT_COUNT").alias("HISTORICAL_WINDOW"),
    F.col("STDEV_FREQUENCY").alias("Rule_1_Frequency_stdev_calculated_value"),
    F.col("MAD_FREQUENCY").alias("Rule_1_Frequency_mad_calculated_value"),
    F.col("MAD_FREQUENCY_ON_AVG").alias("Rule_1_Frequency_mad_on_avg_calculated_value"),    # Added later for the avg calculation
    F.col("STDEV_VALUE").alias("Rule_1_Value_stdev_calculated_value"),
    F.col("MAD_VALUE").alias("Rule_1_Value_mad_calculated_value"),
    F.col("STDEV_QUANTITY").alias("Rule_1_Quantity_stdev_calculated_value"),
    F.col("MAD_QUANTITY").alias("Rule_1_Quantity_mad_calculated_value"),
    F.col("STDEV_MIX").alias("Rule_1_Mix_stdev_calculated_value"),
    F.col("MAD_MIX").alias("Rule_1_Mix_mad_calculated_value"),
    F.col("NUM_SAME_DIRECTION_AVG_FREQUENCY").alias("Rule_2_Frequency_avg_calculated_value"),
    F.col("NUM_SAME_DIRECTION_MEDIAN_FREQUENCY").alias("Rule_2_Frequency_median_calculated_value"),
    F.col("NUM_SAME_DIRECTION_AVG_VALUE").alias("Rule_2_Value_avg_calculated_value"),
    F.col("NUM_SAME_DIRECTION_MEDIAN_VALUE").alias("Rule_2_Value_median_calculated_value"),
    F.col("NUM_SAME_DIRECTION_AVG_QUANTITY").alias("Rule_2_Quantity_avg_calculated_value"),
    F.col("NUM_SAME_DIRECTION_MEDIAN_QUANTITY").alias("Rule_2_Quantity_median_calculated_value"),
    F.col("NUM_SAME_DIRECTION_AVG_MIX").alias("Rule_2_Mix_avg_calculated_value"),
    F.col("NUM_SAME_DIRECTION_MEDIAN_MIX").alias("Rule_2_Mix_median_calculated_value"),
    F.col("NUM_SAME_2SD_DIRECTION_AVG_FREQUENCY").alias("Rule_3_Frequency_avg_calculated_value"),
    F.col("NUM_SAME_2SD_DIRECTION_MEDIAN_FREQUENCY").alias("Rule_3_Frequency_median_calculated_value"),
    F.col("NUM_SAME_2SD_DIRECTION_AVG_VALUE").alias("Rule_3_Value_avg_calculated_value"),
    F.col("NUM_SAME_2SD_DIRECTION_MEDIAN_VALUE").alias("Rule_3_Value_median_calculated_value"),
    F.col("NUM_SAME_2SD_DIRECTION_AVG_QUANTITY").alias("Rule_3_Quantity_avg_calculated_value"),
    F.col("NUM_SAME_2SD_DIRECTION_MEDIAN_QUANTITY").alias("Rule_3_Quantity_median_calculated_value"),
    F.col("NUM_SAME_2SD_DIRECTION_AVG_MIX").alias("Rule_3_Mix_avg_calculated_value"),
    F.col("NUM_SAME_2SD_DIRECTION_MEDIAN_MIX").alias("Rule_3_Mix_median_calculated_value"),
    F.col("PCT_AVG_FREQUENCY").alias("Rule_4_Frequency_avg_calculated_value"),
    F.col("PCT_MEDIAN_FREQUENCY").alias("Rule_4_Frequency_median_calculated_value"),
    F.col("PCT_AVG_VALUE").alias("Rule_4_Value_avg_calculated_value"),
    F.col("PCT_MEDIAN_VALUE").alias("Rule_4_Value_median_calculated_value"),
    F.col("PCT_AVG_QUANTITY").alias("Rule_4_Quantity_avg_calculated_value"),
    F.col("PCT_MEDIAN_QUANTITY").alias("Rule_4_Quantity_median_calculated_value"),
    F.col("PCT_AVG_MIX").alias("Rule_4_Mix_avg_calculated_value"),
    F.col("PCT_MEDIAN_MIX").alias("Rule_4_Mix_median_calculated_value"),
    F.col("PCTILE_AVG_FREQUENCY").alias("Rule_5_Frequency_avg_calculated_value"),
    F.col("PCTILE_MEDIAN_FREQUENCY").alias("Rule_5_Frequency_median_calculated_value"),
    F.col("PCTILE_VALUE").alias("Rule_5_Value_calculated_value"),
    F.col("PCTILE_QUANTITY").alias("Rule_5_Quantity_calculated_value"),
    F.col("PCTILE_AVG_MIX").alias("Rule_5_Mix_calculated_value"),
    "IS_FILLED_ROW"
).orderBy("CUSTOMER_CODE","BASELINE_DATE").where(F.col("CUSTOMER_CODE").isNotNull()) # Remove all of those 0 rows that where only created for calculatin the empty months.
if write:
    df_final_monthly_deviation_table.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(MONTHLY_DEVIATIONS_TABLE)

df_for_ANOMALY_CASE_DETAILS = df_final_monthly_deviation_table  # Just for saving this current table before we overwrite it later
display(df_final_monthly_deviation_table)

# CUSTOMER_RISK_SCORE_MAPPING_TABLE

In [0]:
# # ---------------- Get the bucket table + Function which returns the bucket --------------------

CUSTOMER_RISK_SCORE_MAPPING_TABLE = spark.table("mdl_ui_xpl_dev.ai_sandbox.customer_risk_score_mapping_table")
display(CUSTOMER_RISK_SCORE_MAPPING_TABLE)
CUSTOMER_RISK_SCORE_MAPPING_TABLE = CUSTOMER_RISK_SCORE_MAPPING_TABLE.filter(CUSTOMER_RISK_SCORE_MAPPING_TABLE['metric'] != "RISK_SCORE")   # Remove first row
# Remove "(ABS)" from the 'metric' column values in the mapping table
CUSTOMER_RISK_SCORE_MAPPING_TABLE = CUSTOMER_RISK_SCORE_MAPPING_TABLE.withColumn(
    "metric",
    F.regexp_replace("metric", r"\s*\(ABS\)", "")
)
# Remove "(TOP AND BOTTOM)" from the 'metric' column values in the mapping table
CUSTOMER_RISK_SCORE_MAPPING_TABLE = CUSTOMER_RISK_SCORE_MAPPING_TABLE.withColumn(
    "metric",
    F.regexp_replace("metric", r"\s*\(TOP AND BOTTOM\)", "")
)

# Step 1: Create a dictionary for fast lookup
mapping_dict = {row['metric']: row.asDict() for row in CUSTOMER_RISK_SCORE_MAPPING_TABLE.collect()}

# Step 2: The logic function
def get_bucket_logic(value, metric_key):
    if value is None or metric_key not in mapping_dict:
        return None
    
    row = mapping_dict[metric_key]
    buckets = ["low", "moderate_low", "moderate", "moderate_high", "extremely_high"]
    
    for bucket in buckets:
        rng = str(row.get(bucket, ""))
        if "-" in rng:
            parts = rng.split("-")
            # Handle cases like "0-1" or "-5"
            min_val = float(parts[0]) if parts[0] else float("-inf")
            max_val = float(parts[1]) if parts[1] else float("inf")
            if min_val <= value <= max_val:
                return bucket
            # Add condition for TOP AND BOTTOM with the correct logical and
            if (metric_key == "PCTILE") and ((100 - min_val) >= value >= (100 - max_val)):
                return bucket
    return None

# Step 3: Register as a Spark UDF
get_bucket_udf = F.udf(get_bucket_logic, StringType())

In [0]:
########## Using the function above to change the df_final_monthly_deviation_table values to the correct buckets ###########
STD_MAD = ["Rule_1_Frequency_stdev_calculated_value","Rule_1_Frequency_mad_calculated_value","Rule_1_Frequency_mad_on_avg_calculated_value","Rule_1_Value_stdev_calculated_value","Rule_1_Value_mad_calculated_value","Rule_1_Quantity_stdev_calculated_value","Rule_1_Quantity_mad_calculated_value","Rule_1_Mix_stdev_calculated_value","Rule_1_Mix_mad_calculated_value"]

N_CONSECUTIVE_EVENTS = ["Rule_2_Frequency_avg_calculated_value","Rule_2_Frequency_median_calculated_value","Rule_2_Value_avg_calculated_value","Rule_2_Value_median_calculated_value","Rule_2_Quantity_avg_calculated_value","Rule_2_Quantity_median_calculated_value","Rule_2_Mix_avg_calculated_value","Rule_2_Mix_median_calculated_value"]

N_2SD_DIRECTIO = ["Rule_3_Frequency_avg_calculated_value","Rule_3_Frequency_median_calculated_value","Rule_3_Value_avg_calculated_value","Rule_3_Value_median_calculated_value","Rule_3_Quantity_avg_calculated_value","Rule_3_Quantity_median_calculated_value","Rule_3_Mix_avg_calculated_value","Rule_3_Mix_median_calculated_value"]

PCT_CHANGE = ["Rule_4_Frequency_avg_calculated_value","Rule_4_Frequency_median_calculated_value","Rule_4_Value_avg_calculated_value","Rule_4_Value_median_calculated_value","Rule_4_Quantity_avg_calculated_value","Rule_4_Quantity_median_calculated_value","Rule_4_Mix_avg_calculated_value","Rule_4_Mix_median_calculated_value"]

PCTILE = ["Rule_5_Frequency_avg_calculated_value","Rule_5_Frequency_median_calculated_value","Rule_5_Value_calculated_value","Rule_5_Quantity_calculated_value","Rule_5_Mix_calculated_value"]

In [0]:
# Map your lists to the specific 'metric' name in your mapping table
column_groups = {
    "STD_MAD": STD_MAD,
    "N_CONSECUTIVE_EVENTS": N_CONSECUTIVE_EVENTS,
    "N_2SD_DIRECTIO": N_2SD_DIRECTIO,
    "PCT_CHANGE": PCT_CHANGE,
    "PCTILE": PCTILE
}

for metric_key, columns in column_groups.items():
    for col_name in columns:
        if col_name in df_final_monthly_deviation_table.columns:
            # Use absolute value ONLY for STD_MAD and PCT_CHANGE group
            input_col = F.abs(F.col(col_name)) if metric_key in ["STD_MAD", "PCT_CHANGE"] else F.col(col_name)

            # This creates a new column named 'COLUMN_NAME_bucket'
            df_final_monthly_deviation_table = df_final_monthly_deviation_table.withColumn(
                col_name.replace("calculated_value", "severity_tag"),   # Replace the end to indicate bucket
                get_bucket_udf(input_col, F.lit(metric_key))
            )

# df_final_monthly_deviation_table.select(
#     "*"
# ).where(F.col("CUSTOMER_CODE").isNotNull()) # Remove all of those 0 rows that where only created for calculatin the empty months.
# Show result
display(df_final_monthly_deviation_table)

In [0]:
########### rulesXmetrics table ###########

# List of 20 rulesXmetrics currently used
rulesXmetrics_list = [
    "Rule_1_Frequency_stdev_severity_score",
    "Rule_1_Value_stdev_severity_score",
    "Rule_1_Quantity_stdev_severity_score",
    "Rule_1_Mix_stdev_severity_score",
    "Rule_2_Frequency_avg_severity_score",
    "Rule_2_Value_avg_severity_score",
    "Rule_2_Quantity_avg_severity_score",
    "Rule_2_Mix_avg_severity_score",
    "Rule_3_Frequency_avg_severity_score",
    "Rule_3_Value_avg_severity_score",
    "Rule_3_Quantity_avg_severity_score",
    "Rule_3_Mix_avg_severity_score",
    "Rule_4_Frequency_avg_severity_score",
    "Rule_4_Value_avg_severity_score",
    "Rule_4_Quantity_avg_severity_score",
    "Rule_4_Mix_avg_severity_score",
    "Rule_5_Frequency_avg_severity_score",
    "Rule_5_Value_severity_score",
    "Rule_5_Quantity_severity_score",
    "Rule_5_Mix_severity_score",
]

# Create a table (DataFrame) with the rules and their properties
rulesXmetrics_df = spark.createDataFrame(
    [{"rule_metric": rule, "weight_in_total_severity": 1, "calculate_risk": 1} for rule in rulesXmetrics_list]
)

display(rulesXmetrics_df.select(
    "rule_metric",
    "weight_in_total_severity",
    "calculate_risk"
))

# Optionally, as a dictionary for programmatic use
rulesXmetrics_dict = {
    rule: {"weight_in_total_severity": 1, "calculate_risk": 1}
    for rule in rulesXmetrics_list
}

In [0]:
######## Add severity_scores ########
severity_map = {
    "low": 1,
    "moderate_low": 2,
    "moderate": 3,
    "moderate_high": 4,
    "extremely_high": 5
}

score_exprs = {}
for col_name in df_final_monthly_deviation_table.columns:
    if col_name.endswith("severity_tag"):
        score_col = col_name.replace("severity_tag", "severity_score")
        score_expr = (
            when(F.col(col_name) == "low", 1)
            .when(F.col(col_name) == "moderate_low", 2)
            .when(F.col(col_name) == "moderate", 3)
            .when(F.col(col_name) == "moderate_high", 4)
            .when(F.col(col_name) == "extremely_high", 5)
            .otherwise(None)
        )
        score_exprs[score_col] = score_expr

df_final_monthly_deviation_table = df_final_monthly_deviation_table.withColumns(score_exprs)
# display(df_final_monthly_deviation_table)

In [0]:
################## rules_above_threshold_string ##################

from functools import reduce
from operator import add

# Only use rules in rulesXmetrics_dict with calculate_risk == 1
active_rules = [rule for rule, props in rulesXmetrics_dict.items() if props.get("calculate_risk", 0) == 1]
severity_score_cols = [col for col in df_final_monthly_deviation_table.columns if col in active_rules]
total_number_of_rules = len(severity_score_cols)

# 1. Create a list of the column expressions first, treating null as 0
rule_conditions = [(F.coalesce(F.col(c), F.lit(0)) >= severity_count_threshold).cast("int") for c in severity_score_cols]

# 2. Add them together using reduce
df_final_monthly_deviation_table = df_final_monthly_deviation_table.withColumn(
    "rules_above_threshold",
    reduce(add, rule_conditions)    # sum
).withColumn(
    "rules_above_threshold_string",
    F.concat_ws(
        "/",
        F.col("rules_above_threshold").cast("string"),
        F.lit(str(total_number_of_rules))
    )
)

In [0]:
############### Calculate the overall_risk_score ###############
# Get all severity_score columns that are active in rulesXmetrics_dict
active_severity_score_cols = [
    col for col in df_final_monthly_deviation_table.columns
    if col.endswith("severity_score") and rulesXmetrics_dict.get(col, {}).get("calculate_risk", 0) == 1
]

# Calculate sum and count of non-null columns for each row
sum_expr = reduce(add, [F.coalesce(F.col(c), F.lit(0)) for c in active_severity_score_cols])
count_expr = reduce(add, [(F.col(c).isNotNull()).cast("int") for c in active_severity_score_cols])

df_final_monthly_deviation_table = df_final_monthly_deviation_table.withColumn(
    "overall_risk_score",
    F.when(count_expr > 0, sum_expr / count_expr).otherwise(None)
)

# display(df_final_monthly_deviation_table)

In [0]:
############## Select and Display Final CUSTOMER_RISK_SCORE_TABLE ##############
df_final_monthly_deviation_table = df_final_monthly_deviation_table.select(
    "EVENT_IDENTIFIER",
    "CUSTOMER_CODE",
    "AGGREGATE_BY",
    "BASELINE_DATE",
    "Rule_1_Frequency_stdev_severity_tag",
    #"Rule_1_Frequency_mad_severity_tag",
    #"Rule_1_Frequency_mad_on_avg_severity_tag",
    "Rule_1_Value_stdev_severity_tag",
    #"Rule_1_Value_mad_severity_tag",
    "Rule_1_Quantity_stdev_severity_tag",
    #"Rule_1_Quantity_mad_severity_tag",
    "Rule_1_Mix_stdev_severity_tag",
    #"Rule_1_Mix_mad_severity_tag",
    "Rule_2_Frequency_avg_severity_tag",
    #"Rule_2_Frequency_median_severity_tag",
    "Rule_2_Value_avg_severity_tag",
    #"Rule_2_Value_median_severity_tag",
    "Rule_2_Quantity_avg_severity_tag",
    #"Rule_2_Quantity_median_severity_tag",
    "Rule_2_Mix_avg_severity_tag",
    #"Rule_2_Mix_median_severity_tag",
    "Rule_3_Frequency_avg_severity_tag",
    #"Rule_3_Frequency_median_severity_tag",
    "Rule_3_Value_avg_severity_tag",
    #"Rule_3_Value_median_severity_tag",
    "Rule_3_Quantity_avg_severity_tag",
    #"Rule_3_Quantity_median_severity_tag",
    "Rule_3_Mix_avg_severity_tag",
    #"Rule_3_Mix_median_severity_tag",
    "Rule_4_Frequency_avg_severity_tag",
    #"Rule_4_Frequency_median_severity_tag",
    "Rule_4_Value_avg_severity_tag",
    #"Rule_4_Value_median_severity_tag",
    "Rule_4_Quantity_avg_severity_tag",
    #"Rule_4_Quantity_median_severity_tag",
    "Rule_4_Mix_avg_severity_tag",
    #"Rule_4_Mix_median_severity_tag",
    "Rule_5_Frequency_avg_severity_tag",
    #"Rule_5_Frequency_median_severity_tag",
    "Rule_5_Value_severity_tag",
    "Rule_5_Quantity_severity_tag",
    "Rule_5_Mix_severity_tag",
    
    "Rule_1_Frequency_stdev_severity_score",
    #"Rule_1_Frequency_mad_severity_score",
    #"Rule_1_Frequency_mad_on_avg_severity_score",
    "Rule_1_Value_stdev_severity_score",
    #"Rule_1_Value_mad_severity_score",
    "Rule_1_Quantity_stdev_severity_score",
    #"Rule_1_Quantity_mad_severity_score",
    "Rule_1_Mix_stdev_severity_score",
    #"Rule_1_Mix_mad_severity_score",
    "Rule_2_Frequency_avg_severity_score",
    #"Rule_2_Frequency_median_severity_score",
    "Rule_2_Value_avg_severity_score",
    #"Rule_2_Value_median_severity_score",
    "Rule_2_Quantity_avg_severity_score",
    #"Rule_2_Quantity_median_severity_score",
    "Rule_2_Mix_avg_severity_score",
    #"Rule_2_Mix_median_severity_score",
    "Rule_3_Frequency_avg_severity_score",
    #"Rule_3_Frequency_median_severity_score",
    "Rule_3_Value_avg_severity_score",
    #"Rule_3_Value_median_severity_score",
    "Rule_3_Quantity_avg_severity_score",
    #"Rule_3_Quantity_median_severity_score",
    "Rule_3_Mix_avg_severity_score",
    #"Rule_3_Mix_median_severity_score",
    "Rule_4_Frequency_avg_severity_score",
    #"Rule_4_Frequency_median_severity_score",
    "Rule_4_Value_avg_severity_score",
    #"Rule_4_Value_median_severity_score",
    "Rule_4_Quantity_avg_severity_score",
    #"Rule_4_Quantity_median_severity_score",
    "Rule_4_Mix_avg_severity_score",
    #"Rule_4_Mix_median_severity_score",
    "Rule_5_Frequency_avg_severity_score",
    #"Rule_5_Frequency_median_severity_score",
    "Rule_5_Value_severity_score",
    "Rule_5_Quantity_severity_score",
    "Rule_5_Mix_severity_score",
    "rules_above_threshold_string",
    "overall_risk_score"
)#.where(F.col("CUSTOMER_CODE").isNotNull()) # Remove all of those 0 rows that where only created for calculatin the empty months.
if write:
    df_final_monthly_deviation_table.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(monthly_deviation_table_buckets)
display(df_final_monthly_deviation_table)

In [0]:
############### ANOMALY_LIST table  ################    
# Right now this is generated again in the end, and now only relevent for building the detailed table
from pyspark.sql.functions import current_date

#target_month = "2026-02-01"  # Set your target month (first day of month)

anomaly_list_df = df_final_monthly_deviation_table.filter(
    F.col("BASELINE_DATE") == lit(target_month)
).select(
    F.col("EVENT_IDENTIFIER").alias("anomaly_id"),
    F.lit(1).alias("object_id"),
    F.lit("Order").alias("object_name"),
    F.lit(None).alias("entity_id"),
    F.col("CUSTOMER_CODE").alias("entity_name"),
    F.col("AGGREGATE_BY").alias("Aggregation_level"),
    F.col("overall_risk_score").alias("severity_score"),
    F.lit(None).cast("string").alias("assigned_to"),
    F.lit("New").cast("string").alias("status"),
    F.lit("On-track").cast("string").alias("sla_status"),
    F.current_date().alias("created_at"),
    F.current_date().alias("updated_at"),
    F.lit(None).cast("boolean").alias("feedback_label"),
    F.lit(None).cast("string").alias("feedback_notes"),
    F.col("rules_above_threshold_string").alias("number_of_rules_above_threshold")
)
# if write:
#     anomaly_list_df.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(anomaly_list_table)
# display(anomaly_list_df)

In [0]:
############## ANOMALY_CASE_DETAILS table ##############

a = anomaly_list_df.alias("a")
b = df_for_ANOMALY_CASE_DETAILS.alias("b").filter(F.col("BASELINE_DATE") == target_month)
c = df_final_monthly_deviation_table.alias("c").filter(F.col("BASELINE_DATE") == target_month)

# Join
joined_df = a.join(
    b,
    a.anomaly_id == b.EVENT_IDENTIFIER,
    "left"
).join(
    c,
    a.anomaly_id == c.EVENT_IDENTIFIER,
    "left"
)

# Helper for nulling calculated_value columns if IS_FILLED_ROW == 1
def null_if_filled(col):
    return F.when(F.col("IS_FILLED_ROW") == 1, F.lit(None)).otherwise(col)

# -----------------------------
# Build rules array (clean)
# -----------------------------
rules_array = F.array(

    # RULE 1 - SD
    F.struct(F.lit("RULE 1"), F.lit("Frequency"), F.lit("SD"),
             F.round(F.col("AVG_TOTAL_TIME_FROM_LAST_ORDER"), 2).cast("double"),
             F.round(F.col("HISTORICAL_AVG_TIME_FROM_LAST_ORDER"), 2).cast("double"),
             null_if_filled(F.round(F.col("Rule_1_Frequency_stdev_calculated_value"), 2).cast("double")),
             F.round(F.col("Rule_1_Frequency_stdev_severity_score"), 2).cast("double"), 
             F.col("Rule_1_Frequency_stdev_severity_tag").cast("string")),

    F.struct(F.lit("RULE 1"), F.lit("Value"), F.lit("SD"),
             F.round(F.col("TOTAL_ORDER_VALUE_EUR"), 2).cast("double"),
             F.round(F.col("HISTORICAL_AVG_TOTAL_ORDER_VALUE_EUR"), 2).cast("double"),
             null_if_filled(F.round(F.col("Rule_1_Value_stdev_calculated_value"), 2).cast("double")),
             F.round(F.col("Rule_1_Value_stdev_severity_score")).cast("double"),
             F.col("Rule_1_Value_stdev_severity_tag").cast("string")),
             

    F.struct(F.lit("RULE 1"), F.lit("Quantity"), F.lit("SD"),
             F.round(F.col("TOTAL_ORDER_QUANTITY"), 2).cast("double"),
             F.round(F.col("HISTORICAL_AVG_TOTAL_ORDER_Quantity"), 2).cast("double"),
             null_if_filled(F.round(F.col("Rule_1_Quantity_stdev_calculated_value"), 2).cast("double")),
             F.round(F.col("Rule_1_Quantity_stdev_severity_score")).cast("double"),
             F.col("Rule_1_Quantity_stdev_severity_tag").cast("string")),
             

    F.struct(F.lit("RULE 1"), F.lit("Mix"), F.lit("SD"),
             F.round(F.col("BRAND_FORMAT_MIX_SIMILARITY"), 2).cast("double"),
             F.round(F.col("HISTORICAL_AVG_TOTAL_BRAND_FORMAT_MIX"), 2).cast("double"),
             null_if_filled(F.round(F.col("Rule_1_Mix_stdev_calculated_value"), 2).cast("double")),
             F.round(F.col("Rule_1_Mix_stdev_severity_score")).cast("double"),
             F.col("Rule_1_Mix_stdev_severity_tag").cast("string")),
             

    # RULE 2 - Months
    F.struct(F.lit("RULE 2"), F.lit("Frequency"), F.lit("Months"),
             F.round(F.col("AVG_TOTAL_TIME_FROM_LAST_ORDER"), 2).cast("double"),
             F.round(F.col("HISTORICAL_AVG_TIME_FROM_LAST_ORDER"), 2).cast("double"),
             null_if_filled(F.round(F.col("Rule_2_Frequency_avg_calculated_value"), 2).cast("double")),
             F.round(F.col("Rule_2_Frequency_avg_severity_score")).cast("double"),
             F.col("Rule_2_Frequency_avg_severity_tag").cast("string")),
             

    F.struct(F.lit("RULE 2"), F.lit("Value"), F.lit("Months"),
             F.round(F.col("TOTAL_ORDER_VALUE_EUR"), 2).cast("double"),
             F.round(F.col("HISTORICAL_AVG_TOTAL_ORDER_VALUE_EUR"), 2).cast("double"),
             null_if_filled(F.round(F.col("Rule_2_Value_avg_calculated_value"), 2).cast("double")),
             F.round(F.col("Rule_2_Value_avg_severity_score")).cast("double"),
             F.col("Rule_2_Value_avg_severity_tag").cast("string")),
             

    F.struct(F.lit("RULE 2"), F.lit("Quantity"), F.lit("Months"),
             F.round(F.col("TOTAL_ORDER_QUANTITY"), 2).cast("double"),
             F.round(F.col("HISTORICAL_AVG_TOTAL_ORDER_Quantity"), 2).cast("double"),
             null_if_filled(F.round(F.col("Rule_2_Quantity_avg_calculated_value"), 2).cast("double")),
             F.round(F.col("Rule_2_Quantity_avg_severity_score")).cast("double"),
             F.col("Rule_2_Quantity_avg_severity_tag").cast("string")),
             

    F.struct(F.lit("RULE 2"), F.lit("Mix"), F.lit("Months"),
             F.round(F.col("BRAND_FORMAT_MIX_SIMILARITY"), 2).cast("double"),
             F.round(F.col("HISTORICAL_AVG_TOTAL_BRAND_FORMAT_MIX"), 2).cast("double"),
             null_if_filled(F.round(F.col("Rule_2_Mix_avg_calculated_value"), 2).cast("double")),
             F.round(F.col("Rule_2_Mix_avg_severity_score")).cast("double"),
             F.col("Rule_2_Mix_avg_severity_tag").cast("string")),
             

    # RULE 3 - Months
    F.struct(F.lit("RULE 3"), F.lit("Frequency"), F.lit("Months"),
             F.round(F.col("AVG_TOTAL_TIME_FROM_LAST_ORDER"), 2).cast("double"),
             F.round(F.col("HISTORICAL_AVG_TIME_FROM_LAST_ORDER"), 2).cast("double"),
             null_if_filled(F.round(F.col("Rule_3_Frequency_avg_calculated_value"), 2).cast("double")),
             F.round(F.col("Rule_3_Frequency_avg_severity_score")).cast("double"),
             F.col("Rule_3_Frequency_avg_severity_tag").cast("string")),
             

    F.struct(F.lit("RULE 3"), F.lit("Value"), F.lit("Months"),
             F.round(F.col("TOTAL_ORDER_VALUE_EUR"), 2).cast("double"),
             F.round(F.col("HISTORICAL_AVG_TOTAL_ORDER_VALUE_EUR"), 2).cast("double"),
             null_if_filled(F.round(F.col("Rule_3_Value_avg_calculated_value"), 2).cast("double")),
             F.round(F.col("Rule_3_Value_avg_severity_score")).cast("double"),
             F.col("Rule_3_Value_avg_severity_tag").cast("string")),
             

    F.struct(F.lit("RULE 3"), F.lit("Quantity"), F.lit("Months"),
             F.round(F.col("TOTAL_ORDER_QUANTITY"), 2).cast("double"),
             F.round(F.col("HISTORICAL_AVG_TOTAL_ORDER_Quantity"), 2).cast("double"),
             null_if_filled(F.round(F.col("Rule_3_Quantity_avg_calculated_value"), 2).cast("double")),
             F.round(F.col("Rule_3_Quantity_avg_severity_score")).cast("double"),
             F.col("Rule_3_Quantity_avg_severity_tag").cast("string")),
             

    F.struct(F.lit("RULE 3"), F.lit("Mix"), F.lit("Months"),
             F.round(F.col("BRAND_FORMAT_MIX_SIMILARITY"), 2).cast("double"),
             F.round(F.col("HISTORICAL_AVG_TOTAL_BRAND_FORMAT_MIX"), 2).cast("double"),
             null_if_filled(F.round(F.col("Rule_3_Mix_avg_calculated_value"), 2).cast("double")),
             F.round(F.col("Rule_3_Mix_avg_severity_score")).cast("double"),
             F.col("Rule_3_Mix_avg_severity_tag").cast("string")),
             

    # RULE 4 - Percent
    F.struct(F.lit("RULE 4"), F.lit("Frequency"), F.lit("Percent"),
             F.round(F.col("AVG_TOTAL_TIME_FROM_LAST_ORDER"), 2).cast("double"),
             F.round(F.col("HISTORICAL_AVG_TIME_FROM_LAST_ORDER"), 2).cast("double"),
             null_if_filled(F.round(F.col("Rule_4_Frequency_avg_calculated_value"), 2).cast("double")),
             F.round(F.col("Rule_4_Frequency_avg_severity_score")).cast("double"),
             F.col("Rule_4_Frequency_avg_severity_tag").cast("string")),
             

    F.struct(F.lit("RULE 4"), F.lit("Value"), F.lit("Percent"),
             F.round(F.col("TOTAL_ORDER_VALUE_EUR"), 2).cast("double"),
             F.round(F.col("HISTORICAL_AVG_TOTAL_ORDER_VALUE_EUR"), 2).cast("double"),
             null_if_filled(F.round(F.col("Rule_4_Value_avg_calculated_value"), 2).cast("double")),
             F.round(F.col("Rule_4_Value_avg_severity_score")).cast("double"),
             F.col("Rule_4_Value_avg_severity_tag").cast("string")),
             

    F.struct(F.lit("RULE 4"), F.lit("Quantity"), F.lit("Percent"),
             F.round(F.col("TOTAL_ORDER_QUANTITY"), 2).cast("double"),
             F.round(F.col("HISTORICAL_AVG_TOTAL_ORDER_Quantity"), 2).cast("double"),
             null_if_filled(F.round(F.col("Rule_4_Quantity_avg_calculated_value"), 2).cast("double")),
             F.round(F.col("Rule_4_Quantity_avg_severity_score")).cast("double"),
             F.col("Rule_4_Quantity_avg_severity_tag").cast("string")),
             

    F.struct(F.lit("RULE 4"), F.lit("Mix"), F.lit("Percent"),
             F.round(F.col("BRAND_FORMAT_MIX_SIMILARITY"), 2).cast("double"),
             F.round(F.col("HISTORICAL_AVG_TOTAL_BRAND_FORMAT_MIX"), 2).cast("double"),
             null_if_filled(F.round(F.col("Rule_4_Mix_avg_calculated_value"), 2).cast("double")),
             F.round(F.col("Rule_4_Mix_avg_severity_score")).cast("double"),
             F.col("Rule_4_Mix_avg_severity_tag").cast("string")),
             

    # RULE 5 - Percentile
    F.struct(F.lit("RULE 5"), F.lit("Frequency"), F.lit("Percentile"),
             F.round(F.col("AVG_TOTAL_TIME_FROM_LAST_ORDER"), 2).cast("double"),
             F.round(F.col("HISTORICAL_AVG_TIME_FROM_LAST_ORDER"), 2).cast("double"),
             null_if_filled(F.round(F.col("Rule_5_Frequency_avg_calculated_value"), 2).cast("double")),
             F.round(F.col("Rule_5_Frequency_avg_severity_score")).cast("double"),
             F.col("Rule_5_Frequency_avg_severity_tag").cast("string")),
             

    F.struct(F.lit("RULE 5"), F.lit("Value"), F.lit("Percentile"),
             F.round(F.col("TOTAL_ORDER_VALUE_EUR"), 2).cast("double"),
             F.round(F.col("HISTORICAL_AVG_TOTAL_ORDER_VALUE_EUR"), 2).cast("double"),
             null_if_filled(F.round(F.col("Rule_5_Value_calculated_value"), 2).cast("double")),
             F.round(F.col("Rule_5_Value_severity_score")).cast("double"),
             F.col("Rule_5_Value_severity_tag").cast("string")),
             

    F.struct(F.lit("RULE 5"), F.lit("Quantity"), F.lit("Percentile"),
             F.round(F.col("TOTAL_ORDER_QUANTITY"), 2).cast("double"),
             F.round(F.col("HISTORICAL_AVG_TOTAL_ORDER_Quantity"), 2).cast("double"),
             null_if_filled(F.round(F.col("Rule_5_Quantity_calculated_value"), 2).cast("double")),
             F.round(F.col("Rule_5_Quantity_severity_score")).cast("double"),
             F.col("Rule_5_Quantity_severity_tag").cast("string")),
             

    F.struct(F.lit("RULE 5"), F.lit("Mix"), F.lit("Percentile"),
             F.round(F.col("BRAND_FORMAT_MIX_SIMILARITY"), 2).cast("double"),
             F.round(F.col("HISTORICAL_AVG_TOTAL_BRAND_FORMAT_MIX"), 2).cast("double"),
             null_if_filled(F.round(F.col("Rule_5_Mix_calculated_value"), 2).cast("double")),
             F.round(F.col("Rule_5_Mix_severity_score")).cast("double"),
             F.col("Rule_5_Mix_severity_tag").cast("string"))
)

# -----------------------------
# Explode
# -----------------------------
df_exploded = joined_df.withColumn("rule", F.explode(rules_array))

# -----------------------------
# Final select
# -----------------------------
anomaly_case_details_df = df_exploded.select(
    F.col("anomaly_id").alias("alert_id"),
    F.lit(1).alias("object_id"),
    F.lit(None).alias("rule_id"),
    #F.col("c.BASELINE_DATE").alias("BASELINE_DATE"),
    F.col("c.CUSTOMER_CODE").alias("entity_name"),
    F.lit(None).alias("entity_id"),
    F.col("a.Aggregation_level").alias("entity_level"),
    F.lit(None).alias("metric_id"),
    F.lit("New").alias("status"),
    F.lit("Business").alias("anomaly_type"),
    F.col("created_at").alias("detected_at"),
    F.lit(None).alias("resolved_at"),
    F.lit(None).alias("assigned_to"),

    F.col("rule.col1").alias("rule_name"),
    F.col("rule.col2").alias("metric_name"),
    F.col("rule.col3").alias("unit"),
    F.col("rule.col4").alias("actual_value"),
    F.col("rule.col5").alias("expected_value"),
    F.col("rule.col6").alias("calculated_rule"),
    F.col("rule.col7").alias("severity_score"),
    F.col("rule.col8").alias("severity"),
    F.lit(None).alias("severity_feedback"),
    F.lit(None).alias("Additional_value"),
    (1 - F.col("IS_FILLED_ROW")).alias("had_order") # NOT, 1 = had order on this month, 0 = empty month
)

# # write if needed
# if write:
#     anomaly_case_details_df.write.mode("overwrite") \
#         .option("overwriteSchema", "true") \
#         .saveAsTable(anomaly_case_details_table)

#display(anomaly_case_details_df)

## pull product tables

In [0]:


product_alert_table = spark.table("mdl_ui_xpl_dev.ai_sandbox.alerts_output_table").withColumnRenamed("metric", "metric_name").withColumn("metric_id", lit(None).cast("string"))
price_alert_table = spark.table("mdl_ui_xpl_dev.ai_sandbox.u2k2_uapl_price_correctness_q1_alerts_output_table")
# display(price_alert_table)
# print("Distinct entity_name count in product_alert_table:", product_alert_table.select("entity_name").distinct().count())
# print("Total row count in product_alert_table:", product_alert_table.count())
union_df = anomaly_case_details_df.unionByName(product_alert_table, allowMissingColumns=True).unionByName(price_alert_table, allowMissingColumns=True)
# write if needed
if write:
    union_df.write.mode("overwrite") \
        .option("overwriteSchema", "true") \
        .saveAsTable(union_anomaly_case_details_table)
display(union_df)


# ordered_cols = [
#     "alert_id",
#     "object_id",
#     "rule_id",
#     "entity_name",
#     "entity_id",
#     "entity_level",
#     "metric_id",
#     "status",
#     "anomaly_type",
#     "detected_at",
#     "resolved_at",
#     "assigned_to",
#     "rule_name",
#     "metric_name",
#     "unit",
#     "actual_value",
#     "expected_value",
#     "calculated_rule",
#     "Additional_value",
#     "had_order"
# ]

In [0]:
################################################ 
# Group by entity_name and aggregate as anomaly_list_df
grouped_anomaly_list_df = union_df.groupBy("entity_name").agg(
    F.first("alert_id").alias("anomaly_id"),    
    F.first("object_id").alias("object_id"),
    F.when(F.first("object_id") == 1, "Sales")
     .when(F.first("object_id") == 2, "Product")
     .when(F.first("object_id") == 3, "Price")
     .otherwise("").alias("object_name"),
    F.first("entity_id").alias("entity_id"),    # Supposed to aggregate by this
    #F.first("entity_name").alias("entity_name"),    # Aggragation
    #F.first("entity_level").alias("Aggregation_level"),
    F.avg("severity_score").alias("severity_score"), # for product and price put 1
    F.lit(None).cast("string").alias("assigned_to"),
    F.lit("New").cast("string").alias("status"),    # New/In-progress/Solved
    F.lit("On-track").cast("string").alias("sla_status"),   # On-track/SLA Alert/OverSLA
    F.current_date().alias("created_at"),
    F.current_date().alias("updated_at"),
    F.lit(True).cast("boolean").alias("feedback_label"),
    F.lit(None).cast("string").alias("feedback_notes"),
    F.sum(
        F.when(
            (F.col("object_id") == 1) & (F.col("severity_score") >= severity_count_threshold), 1
        ).when(
            (F.col("object_id") != 1) & (F.col("severity_score").isNotNull()), 1
        ).otherwise(0)
    ).alias("number_of_rules_above_threshold")
)

grouped_anomaly_list_df = grouped_anomaly_list_df.select(
    "anomaly_id",
    "object_id",
    "object_name",
    "entity_id",
    "entity_name",
    "severity_score",
    "assigned_to",
    "status",
    "sla_status",
    "created_at",
    "updated_at",
    "feedback_label",
    "feedback_notes",
    "number_of_rules_above_threshold"
).orderBy(F.col("object_id").asc(), F.col("anomaly_id"))
if write:
    grouped_anomaly_list_df.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(anomaly_list_table)
display(grouped_anomaly_list_df)